# Kitchener Water Main Failure Prediction — Climate-Integrated Analysis

**Author:** Hafsa  
**Institution:** MSc GRA Application — UrbanLinks Lab, Concordia University  
**Supervisor (target):** Dr. Rebecca Dziedzic  

## Study Overview

This notebook implements a complete climate-integrated water main failure prediction pipeline for Kitchener, Ontario. The analysis extends the Kitchener case study of Khashei et al. (2024) from decade-level binary classification to monthly count regression with CMIP6 climate projections to 2100.

**Research Questions:**
1. How will projected climate change affect water main break frequency in Kitchener through 2100?
2. Which pipe materials are most climate-sensitive?
3. How does projected climate change affect the proportion of DI pipes that experience a first break before their design life?

## Pipeline Structure

| Step | Script Section | Output |
|---|---|---|
| 1 | Break Panel | `monthly_panel_primary.csv` |
| 2 | Climate Processing | `climate_monthly.csv` |
| 3 | Dataset Merge | `modelling_dataset.csv` |
| 4 | Negative Binomial Models | `model_results_both_splits.csv` |
| 5 | CMIP6 Processing | `cmip6_monthly_*.csv` |
| 6 | Climate Projections | `projections_annual.csv` |
| 7 | DI Pipe-Level Analysis | `di_pipe_level_coefficients.csv` |
| 8 | Design Life Analysis | `design_life_results.csv` |

## Required Input Files

Place these in the same folder before running:
- `Water_Main_Breaks.csv` — City of Kitchener open data
- `Water_Mains.csv` — City of Kitchener open data  
- All ECCC climate station CSVs (climate-daily*.csv, WATERLOO*.csv, KITCHENER*.csv)
- All CMIP6 NetCDF files (*.nc) from PCIC CanDCS-M6


## Dependencies

In [ ]:
# Install required packages
!pip install netCDF4 statsmodels scipy pandas numpy matplotlib --quiet

---
## Step 1: Build Monthly Break Panel

Loads raw break records and pipe inventory, filters to 1997–2025 (systematic logging start confirmed by data audit), and builds a monthly panel of break counts and exposure (km in service) for CI, DI, and PVC.

**Key decision:** Study period begins 1997, not 1985. Raw data contains only 10 breaks across 1985–1996 versus 99 in 1997 alone — a logging artefact confirmed by visual inspection.


In [ ]:
# -*- coding: utf-8 -*-
"""Kitchener Panel final.ipynb

Automatically generated by Colab.

Original file is located at
    https://colab.research.google.com/drive/1VA_gGgllKvn0TDFvyuEiDEgaDDZHCqL3
"""

import os
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

BREAKS_FILE = "Water_Main_Breaks.csv"
MAINS_FILE  = "Water_Mains.csv"

OUTPUT_PRIMARY     = "monthly_panel_primary.csv"
OUTPUT_SENSITIVITY = "monthly_panel_sensitivity.csv"

STUDY_START    = "1997-01"
STUDY_END      = "2025-12"
CORE_MATERIALS = ["CI", "DI", "PVC"]

# Mean segment length per material from current inventory (metres)
# Used ONLY for the sensitivity analysis — not the primary model
MEAN_SEGMENT_M = {"CI": 75.1, "DI": 63.2, "PVC": 47.4}

print("Loading data...")

breaks = pd.read_csv(BREAKS_FILE, encoding="utf-8-sig", low_memory=False)
mains  = pd.read_csv(MAINS_FILE,  encoding="utf-8-sig", low_memory=False)

print(f"  Breaks loaded : {len(breaks):,} rows")
print(f"  Mains loaded  : {len(mains):,} rows")

print("\nCleaning break records...")

breaks["incident_dt"]  = pd.to_datetime(breaks["Incident date"], errors="coerce")
breaks["year"]         = breaks["incident_dt"].dt.year
breaks["month_period"] = breaks["incident_dt"].dt.to_period("M")

# MAIN breaks only (not service connections)
breaks = breaks[breaks["Type of Asset Broken"] == "MAIN"].copy()

# Complete study years only
breaks = breaks[(breaks["year"] >= 1997) & (breaks["year"] <= 2025)].copy()

# Three core materials only
breaks = breaks[breaks["Asset Material"].isin(CORE_MATERIALS)].copy()

# Must have a valid date
breaks = breaks[breaks["incident_dt"].notna()].copy()

print(f"  Breaks after cleaning : {len(breaks):,}")
print(f"  Material breakdown:")
print(breaks["Asset Material"].value_counts().to_string(header=False))

# Flag retired pipes — used for sensitivity only
n_retired = (breaks["Asset Exists"] == "N").sum()
print(f"\n  Breaks on retired pipes (Asset Exists=N): {n_retired:,}"
      f" ({100*n_retired/len(breaks):.1f}%)")
print("  These are excluded from the primary exposure but included"
      " in the sensitivity check.")

print("\nCleaning mains inventory...")

mains["install_dt"] = pd.to_datetime(mains["INSTALLATION_DATE"], errors="coerce")
mains["install_yr"] = mains["install_dt"].dt.year

mains = mains[mains["MATERIAL"].isin(CORE_MATERIALS)].copy()
mains = mains[mains["Shape__Length"] > 0].copy()
mains["length_km"] = mains["Shape__Length"] / 1000.0

print(f"  Mains after cleaning : {len(mains):,}")
print(f"  Total length         : {mains['length_km'].sum():.1f} km")
print(f"  Material breakdown (km):")
print(mains.groupby("MATERIAL")["length_km"].sum().round(1).to_string())
print(f"\n  Mean segment length by material (m) — used for sensitivity only:")
print(mains.groupby("MATERIAL")["Shape__Length"].mean().round(1).to_string())

monthly_index = pd.period_range(start=STUDY_START, end=STUDY_END, freq="M")
n_months = len(monthly_index)
print(f"\nStudy window: {STUDY_START} to {STUDY_END} ({n_months} months)")

print("\nBuilding monthly exposure...")

primary_rows = []
for mat in CORE_MATERIALS:
    mat_pipes = mains[mains["MATERIAL"] == mat]
    for month in monthly_index:
        yr = month.year
        km = mat_pipes.loc[mat_pipes["install_yr"] <= yr, "length_km"].sum()
        primary_rows.append({"material": mat, "month": month, "km_primary": km})

exposure_primary = pd.DataFrame(primary_rows)

# Get one record per retired pipe: material, install_yr, last_break_yr
retired_breaks = breaks[
    (breaks["Asset Exists"] == "N") &
    (breaks["Asset Material"].isin(CORE_MATERIALS))
].copy()

retired_breaks["install_yr_b"] = pd.to_numeric(
    retired_breaks["Year Asset Installed"], errors="coerce"
)

# Group by material + install_yr + road segment to identify unique retired pipes
retired_pipes = (
    retired_breaks[retired_breaks["install_yr_b"].notna()]
    .groupby(["Asset Material", "install_yr_b", "Road Segment ID"])
    .agg(last_break_yr=("year", "max"))
    .reset_index()
    .rename(columns={"Asset Material": "material",
                     "install_yr_b":   "install_yr"})
)

# Estimated retirement year = year after last known break
retired_pipes["retire_yr"] = retired_pipes["last_break_yr"] + 1

# Approximate length = mean segment length for that material
retired_pipes["length_km"] = (
    retired_pipes["material"].map(MEAN_SEGMENT_M) / 1000.0
)

# Sanity: clamp install years
retired_pipes = retired_pipes[
    (retired_pipes["install_yr"] >= 1880) &
    (retired_pipes["install_yr"] <= 2025)
].copy()

print(f"  Retired pipes reconstructed (for sensitivity): {len(retired_pipes):,}")
print(f"  Breakdown by material:")
print(retired_pipes["material"].value_counts().to_string(header=False))

# Build sensitivity exposure
sensitivity_rows = []
for mat in CORE_MATERIALS:
    mat_ret  = retired_pipes[retired_pipes["material"] == mat]
    # Primary km already computed above — look it up
    prim_mat = exposure_primary[exposure_primary["material"] == mat].set_index("month")

    for month in monthly_index:
        yr = month.year
        km_prim = prim_mat.loc[month, "km_primary"]

        # Add km from retired pipes active during this year
        km_ret = mat_ret.loc[
            (mat_ret["install_yr"] <= yr) & (mat_ret["retire_yr"] > yr),
            "length_km"
        ].sum()

        sensitivity_rows.append({
            "material":    mat,
            "month":       month,
            "km_primary":  km_prim,
            "km_retired":  km_ret,
            "km_sensitivity": km_prim + km_ret
        })

exposure_sensitivity = pd.DataFrame(sensitivity_rows)

print(f"\n  Average km in service by material:")
print(f"  {'Material':<10} {'Primary (snapshot)':>20} {'Sensitivity (+retired)':>24} {'Diff %':>8}")
for mat in CORE_MATERIALS:
    p = exposure_primary[exposure_primary["material"]==mat]["km_primary"].mean()
    s = exposure_sensitivity[exposure_sensitivity["material"]==mat]["km_sensitivity"].mean()
    print(f"  {mat:<10} {p:>20.1f} {s:>24.1f} {100*(s-p)/p:>7.2f}%")

print("\n  → If Diff% is small (<5%), the correction does not materially")
print("    affect results and can be safely set aside (report in paper).")

#COUNT BREAKS PER MATERIAL PER MONTH
print("\nCounting breaks per material per month...")

break_counts = (
    breaks
    .groupby(["Asset Material", "month_period"])
    .size()
    .reset_index(name="break_count")
    .rename(columns={"Asset Material": "material", "month_period": "month"})
)

# Full grid so zero months are explicit, not missing
full_grid = pd.MultiIndex.from_product(
    [CORE_MATERIALS, monthly_index], names=["material", "month"]
).to_frame(index=False)

break_counts = full_grid.merge(break_counts, on=["material", "month"], how="left")
break_counts["break_count"] = break_counts["break_count"].fillna(0).astype(int)

print(f"  Total breaks by material:")
print(break_counts.groupby("material")["break_count"].sum().to_string())
print(f"\n  Zero-count months by material:")
zero = break_counts[break_counts["break_count"] == 0].groupby("material").size()
print((zero.astype(str) + " / " + str(n_months)).to_string())

def build_panel(break_counts, exposure_df, km_col):
    panel = break_counts.merge(exposure_df, on=["material", "month"])
    panel["failure_rate"] = np.where(
        panel[km_col] > 0,
        panel["break_count"] / panel[km_col],
        np.nan
    )
    panel["year"]      = panel["month"].dt.year
    panel["month_num"] = panel["month"].dt.month
    panel["sin_month"] = np.sin(2 * np.pi * panel["month_num"] / 12)
    panel["cos_month"] = np.cos(2 * np.pi * panel["month_num"] / 12)
    panel["time_index"]= (panel["year"] - 1997) * 12 + panel["month_num"] - 1
    panel["month_str"] = panel["month"].astype(str)
    return panel.sort_values(["material", "month_str"]).reset_index(drop=True)


print("\nBuilding primary panel...")
panel_primary = build_panel(
    break_counts,
    exposure_primary[["material", "month", "km_primary"]],
    "km_primary"
)
panel_primary = panel_primary[[
    "material", "month_str", "year", "month_num",
    "break_count", "km_primary", "failure_rate",
    "sin_month", "cos_month", "time_index"
]]

print("Building sensitivity panel...")
panel_sensitivity = build_panel(
    break_counts,

---
## Step 2: Build Monthly Climate Covariates

Merges three ECCC climate stations by priority (KITCHENER/WATERLOO primary, REGION OF WATERLOO INT'L AIRPORT 2002–2010, WATERLOO WELLINGTON A 1996–2002), fills gaps by linear interpolation (max 5 consecutive days), and calculates monthly climate covariates.

**Covariates produced:** FI (freezing index), FD (freezing days), FTC (freeze-thaw cycles), FI_cum (cumulative FI since October), Tmean, Tmin, Tmax, ADD, TIG, TDG.

**Note:** Precipitation excluded due to measurement inconsistency between stations after 2010 (KITCHENER/WATERLOO station reports TOTAL_PRECIPITATION but not TOTAL_RAIN reliably).


In [ ]:
    exposure_sensitivity[["material", "month", "km_primary",
                           "km_retired", "km_sensitivity"]],
    "km_sensitivity"
)
panel_sensitivity = panel_sensitivity[[
    "material", "month_str", "year", "month_num",
    "break_count", "km_primary", "km_retired", "km_sensitivity",
    "failure_rate", "sin_month", "cos_month", "time_index"
]]

script_dir = os.getcwd()

path_primary     = os.path.join(script_dir, OUTPUT_PRIMARY)
path_sensitivity = os.path.join(script_dir, OUTPUT_SENSITIVITY)

panel_primary.to_csv(path_primary, index=False)
panel_sensitivity.to_csv(path_sensitivity, index=False)

print(f"\n✓ Primary panel saved     : {path_primary}")
print(f"  Shape: {panel_primary.shape}")
print(f"\n✓ Sensitivity panel saved : {path_sensitivity}")
print(f"  Shape: {panel_sensitivity.shape}")

print(f"\n  Columns in primary panel:")
for col in panel_primary.columns:
    print(f"    {col}")

STUDY_START = "1996-01-01"
STUDY_END   = "2025-12-31"

# Max consecutive missing days to fill by interpolation
# Beyond this, the gap is too large to reliably fill
MAX_GAP_DAYS = 5

# 1981-2010 monthly normals from WATERLOO WELLINGTON A (climate-normals.csv)
# Used ONLY for rainfall deficit (RD) calculation
NORMAL_RAINFALL_MM = {
    1: 28.66, 2: 29.74, 3: 36.81,  4: 67.98,
    5: 81.80, 6: 82.39, 7: 98.55,  8: 83.92,
    9: 87.79, 10: 66.09, 11: 75.02, 12: 38.02
}

# Station priority: lower number = preferred
STATION_PRIORITY = {
    'KITCHENER/WATERLOO':                  1,
    "REGION OF WATERLOO INT'L AIRPORT":    2,
    'WATERLOO WELLINGTON A':               3,
    'WATERLOO WELLINGTON 2':               4,
    'WATERLOO WPCP':                       5,
}

OUTPUT_DAILY   = "climate_daily_merged.csv"
OUTPUT_MONTHLY = "climate_monthly.csv"

import os
import glob
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

print("=" * 60)
print("STEP 1: Loading climate files")
print("=" * 60)

try:
    script_dir = os.path.dirname(os.path.abspath(__file__))
except NameError:
    script_dir = os.getcwd()

# Find all climate CSV files - user should put them all in same folder
climate_files = glob.glob(os.path.join(script_dir, "climate-daily*.csv")) + \
                glob.glob(os.path.join(script_dir, "WATERLOO*.csv"))     + \
                glob.glob(os.path.join(script_dir, "KITCHENER*.csv"))

if not climate_files:
    raise FileNotFoundError(
        "No climate files found in current folder.\n"
        "Make sure all climate-daily*.csv, WATERLOO*.csv, "
        "and KITCHENER*.csv files are in the same folder as this script."
    )

print(f"  Found {len(climate_files)} climate files:")
for f in sorted(climate_files):
    print(f"    {os.path.basename(f)}")

all_dfs = []
for f in climate_files:
    try:
        df = pd.read_csv(f, encoding='utf-8-sig', low_memory=False)
        all_dfs.append(df)
    except Exception as e:
        print(f"  WARNING: Could not read {os.path.basename(f)}: {e}")

combined = pd.concat(all_dfs, ignore_index=True)
print(f"\n  Total rows loaded: {len(combined):,}")

print("\n" + "=" * 60)
print("STEP 2: Parsing dates and filtering to study window")
print("=" * 60)

combined['date'] = pd.to_datetime(combined['LOCAL_DATE'], errors='coerce')
combined = combined[combined['date'].notna()].copy()
combined = combined[
    (combined['date'] >= STUDY_START) &
    (combined['date'] <= STUDY_END)
].copy()
combined['year']  = combined['date'].dt.year
combined['month'] = combined['date'].dt.month
combined['day']   = combined['date'].dt.day

print(f"  Rows in study window 1996-2025: {len(combined):,}")
print(f"  Stations present:")
for stn, grp in combined.groupby('STATION_NAME'):
    print(f"    {stn}: {grp['date'].dt.year.min():.0f}–"
          f"{grp['date'].dt.year.max():.0f} "
          f"({len(grp):,} rows)")

print("\n" + "=" * 60)
print("STEP 3: Selecting best station per day")
print("=" * 60)

combined['priority'] = combined['STATION_NAME'].map(STATION_PRIORITY).fillna(99)

# Sort by date then priority so the best station comes first
combined_sorted = combined.sort_values(['date', 'priority'])

# For each day keep only the row from the highest-priority station
# that has actual temperature data
# First try: keep rows with MEAN_TEMPERATURE not null
has_temp = combined_sorted[combined_sorted['MEAN_TEMPERATURE'].notna()]
best_with_temp = has_temp.drop_duplicates('date', keep='first')

# For days with no temperature at all, keep the best-priority row anyway
# (we will gap-fill later)
all_days_best = combined_sorted.drop_duplicates('date', keep='first')

# Merge: use temp-row where available, fallback row otherwise
daily = all_days_best.copy()
temp_dates = set(best_with_temp['date'])
daily = daily[~daily['date'].isin(temp_dates)]
daily = pd.concat([best_with_temp, daily], ignore_index=True)
daily = daily.sort_values('date').reset_index(drop=True)

print(f"  Unique days after merge: {len(daily):,}")
print(f"  Days with MEAN_TEMPERATURE: {daily['MEAN_TEMPERATURE'].notna().sum():,}")
print(f"  Days with MIN_TEMPERATURE:  {daily['MIN_TEMPERATURE'].notna().sum():,}")
print(f"  Days with MAX_TEMPERATURE:  {daily['MAX_TEMPERATURE'].notna().sum():,}")
print(f"  Days with TOTAL_RAIN:       {daily['TOTAL_RAIN'].notna().sum():,}")

print("\n" + "=" * 60)
print("STEP 4: Building complete date spine and filling gaps")
print("=" * 60)

# Create a row for every single day in the study window
spine = pd.DataFrame({
    'date': pd.date_range(start=STUDY_START, end=STUDY_END, freq='D')
})

# Merge our best-station data onto the spine
# Days where no station had any data will be NaN
daily_full = spine.merge(
    daily[['date', 'STATION_NAME', 'priority',
           'MEAN_TEMPERATURE', 'MIN_TEMPERATURE', 'MAX_TEMPERATURE',
           'TOTAL_RAIN', 'TOTAL_PRECIPITATION', 'TOTAL_SNOW']],
    on='date', how='left'
)

daily_full['year']  = daily_full['date'].dt.year
daily_full['month'] = daily_full['date'].dt.month
daily_full['day']   = daily_full['date'].dt.day

total_days = len(daily_full)
print(f"  Total days in spine: {total_days:,}")

# Flag missing days BEFORE gap filling (so we can count them per month)
daily_full['was_missing'] = daily_full['MEAN_TEMPERATURE'].isna().astype(int)
missing_total = daily_full['was_missing'].sum()
print(f"  Days with missing MEAN_TEMPERATURE: {missing_total} ({100*missing_total/total_days:.1f}%)")

# Gap fill by linear interpolation, but ONLY for gaps <= MAX_GAP_DAYS
# Longer gaps are left as NaN and flagged
for col in ['MEAN_TEMPERATURE', 'MIN_TEMPERATURE', 'MAX_TEMPERATURE',
            'TOTAL_RAIN', 'TOTAL_PRECIPITATION', 'TOTAL_SNOW']:
    daily_full[col] = (
        daily_full[col]
        .interpolate(method='linear', limit=MAX_GAP_DAYS,
                     limit_direction='both')
    )

filled = daily_full['MEAN_TEMPERATURE'].notna().sum()
print(f"  Days with MEAN_TEMPERATURE after gap-fill (max {MAX_GAP_DAYS} days): "
      f"{filled:,} ({100*filled/total_days:.1f}%)")

still_missing = daily_full['MEAN_TEMPERATURE'].isna().sum()
if still_missing > 0:
    print(f"  WARNING: {still_missing} days still missing after interpolation.")
    print("  These are gaps longer than 5 days. Monthly values for those months")
    print("  will be based on fewer days than normal.")
    print("  Missing by year:")
    miss_yr = daily_full[daily_full['MEAN_TEMPERATURE'].isna()].groupby('year').size()
    print(miss_yr.to_string())

def calc_gradients(group):
    """
    For a month's daily Tmean series, calculate:
    TIG = max rate of temperature increase over any pair of consecutive days
    TDG = max rate of temperature decrease over any pair of consecutive days
    Per Khashei et al. (2024): TIG = max{(T_k - T_j)/(k-j)} for j < k
    We use consecutive days only (k-j=1) which is the most common interpretation
    and avoids over-smoothing.
    """
    t = group['MEAN_TEMPERATURE'].dropna().values
    if len(t) < 2:
        return pd.Series({'TIG': np.nan, 'TDG': np.nan})
    diffs = np.diff(t)
    TIG = float(np.max(diffs))   if len(diffs) > 0 else np.nan
    TDG = float(np.max(-diffs))  if len(diffs) > 0 else np.nan
    return pd.Series({'TIG': TIG, 'TDG': TDG})

def calc_ftc(group):
    """
    Freeze-thaw cycle: a day where Tmin < 0 AND Tmax > 0.
    This means the temperature crossed zero in both directions on the same day.
    """
    ftc = ((group['MIN_TEMPERATURE'] < 0) & (group['MAX_TEMPERATURE'] > 0)).sum()
    return int(ftc)

def calc_fi_cumulative(df_year, year, month):
    """
    FI_cum = cumulative freezing index since the start of the freeze season.
    Freeze season starts in October of the previous year.
    So for Jan 2010, we sum FI from Oct 2009 through Jan 2010.
    """
    # Freeze season start: October of previous year if month < 10, else October of this year
    if month < 10:
        season_start = pd.Timestamp(year=year-1, month=10, day=1)
    else:
        season_start = pd.Timestamp(year=year, month=10, day=1)
    season_end = pd.Timestamp(year=year, month=month, day=1) + pd.offsets.MonthEnd(0)
    mask = (df_year['date'] >= season_start) & (df_year['date'] <= season_end)
    fi_cum = df_year.loc[mask, 'MEAN_TEMPERATURE'].apply(lambda x: min(x, 0) if pd.notna(x) else 0).sum()
    return round(fi_cum, 2)

monthly_rows = []

for (yr, mo), grp in daily_full.groupby(['year', 'month']):
    grp = grp.copy()
    n_days       = len(grp)
    n_valid_temp = grp['MEAN_TEMPERATURE'].notna().sum()
    n_missing    = int(grp['was_missing'].sum())

    # Skip month if more than half the days are still missing
    if n_valid_temp < n_days / 2:
        print(f"  WARNING: {yr}-{mo:02d} has only {n_valid_temp}/{n_days} valid "
              f"temp days — covariates will be unreliable")

    # Temperature level covariates
    Tmean = grp['MEAN_TEMPERATURE'].mean()
    Tmin  = grp['MIN_TEMPERATURE'].mean()
    Tmax  = grp['MAX_TEMPERATURE'].mean()
    ADD   = (grp['MAX_TEMPERATURE'] - grp['MIN_TEMPERATURE']).mean()

    # Frost covariates
    FI  = grp['MEAN_TEMPERATURE'].apply(lambda x: min(x, 0) if pd.notna(x) else 0).sum()
    FD  = (grp['MEAN_TEMPERATURE'] < 0).sum()
    FTC = calc_ftc(grp)

    # Thaw covariates
    TI  = grp['MEAN_TEMPERATURE'].apply(lambda x: max(x, 0) if pd.notna(x) else 0).sum()
    TD  = (grp['MEAN_TEMPERATURE'] > 0).sum()

    # Temperature gradient covariates
    grads = calc_gradients(grp)
    TIG   = grads['TIG']
    TDG   = grads['TDG']

    # Cumulative freezing index since freeze season start
    FI_cum = calc_fi_cumulative(daily_full, yr, mo)

    # Precipitation covariates
    RI   = grp['TOTAL_RAIN'].sum()
    PREC = grp['TOTAL_PRECIPITATION'].sum()

    # Rainfall deficit vs 1981-2010 normal
    RD   = round(RI - NORMAL_RAINFALL_MM.get(mo, np.nan), 2)

    monthly_rows.append({
        'year':        yr,
        'month':       mo,
        'month_str':   f"{yr}-{mo:02d}",
        'n_days':      n_days,
        'n_valid_temp':n_valid_temp,
        'missing_days':n_missing,
        # Temperature
        'Tmean':   round(Tmean, 3) if pd.notna(Tmean) else np.nan,
        'Tmin':    round(Tmin,  3) if pd.notna(Tmin)  else np.nan,
        'Tmax':    round(Tmax,  3) if pd.notna(Tmax)  else np.nan,
        'ADD':     round(ADD,   3) if pd.notna(ADD)   else np.nan,
        # Frost
        'FI':      round(FI,    2),
        'FD':      int(FD),
        'FTC':     int(FTC),
        'FI_cum':  FI_cum,
        # Thaw
        'TI':      round(TI, 2),
        'TD':      int(TD),
        # Gradient
        'TIG':     round(TIG, 3) if pd.notna(TIG) else np.nan,
        'TDG':     round(TDG, 3) if pd.notna(TDG) else np.nan,
        # Precipitation
        'RI':      round(RI,   2),
        'RD':      RD,
        'PREC':    round(PREC, 2),
    })

monthly = pd.DataFrame(monthly_rows)
monthly = monthly.sort_values(['year','month']).reset_index(drop=True)

print(f"\n  Monthly covariate table shape: {monthly.shape}")
print(f"  Rows: {len(monthly)} (expect 360 for 1996-2025)")
print(f"\n  Sample — first 6 months:")
print(monthly.head(6)[['month_str','Tmean','FI','FD','FTC','FI_cum','RI','RD']].to_string(index=False))

print("\n" + "=" * 60)
print("STEP 6: Sanity checks")
print("=" * 60)

# Check 1: FI should be highest in Jan/Feb
print("\n  Mean FI by month (expect highest in Jan/Feb):")
print(monthly.groupby('month')['FI'].mean().round(1).to_string())

# Check 2: FTC should peak in March/April (transition season)
print("\n  Mean FTC by month (expect peak in Mar/Apr):")
print(monthly.groupby('month')['FTC'].mean().round(1).to_string())

# Check 3: Tmean should be negative Nov-Mar
print("\n  Mean Tmean by month (expect negative Nov-Mar):")
print(monthly.groupby('month')['Tmean'].mean().round(1).to_string())

# Check 4: RI should be highest in summer
print("\n  Mean RI by month (expect highest Jun-Sep):")
print(monthly.groupby('month')['RI'].mean().round(1).to_string())

# Check 5: Missing data flag
total_missing = monthly['missing_days'].sum()
print(f"\n  Total gap-filled days across all months: {total_missing}")
print(f"  Months with any gap-filled days: {(monthly['missing_days']>0).sum()}")
high_missing = monthly[monthly['missing_days'] > 5]
if len(high_missing) > 0:
    print(f"  Months with >5 gap-filled days (flag these in paper):")
    print(high_missing[['month_str','missing_days']].to_string(index=False))

# Check 6: No NaN in key covariates
print("\n  NaN counts in key covariates:")
key_cols = ['Tmean','Tmin','Tmax','FI','FD','FTC','TI','TD','TIG','TDG','FI_cum']
for col in key_cols:
    n = monthly[col].isna().sum()
    if n > 0:
        print(f"    {col}: {n} NaN values")
    else:
        print(f"    {col}: OK")

print("\n" + "=" * 60)
print("STEP 7: Saving outputs")
print("=" * 60)

daily_out_path   = os.path.join(script_dir, OUTPUT_DAILY)
monthly_out_path = os.path.join(script_dir, OUTPUT_MONTHLY)

# Daily merged file
daily_full[['date','year','month','day','STATION_NAME',
            'MEAN_TEMPERATURE','MIN_TEMPERATURE','MAX_TEMPERATURE',
            'TOTAL_RAIN','TOTAL_PRECIPITATION','TOTAL_SNOW',
            'was_missing']].to_csv(daily_out_path, index=False)

# Monthly covariate file
monthly.to_csv(monthly_out_path, index=False)

print(f"\n  Daily merged file  : {daily_out_path}")
print(f"  Rows: {len(daily_full):,}  |  Columns: 12")
print(f"\n  Monthly covariate file: {monthly_out_path}")
print(f"  Rows: {len(monthly):,}  |  Columns: {len(monthly.columns)}")
print(f"\n  Columns in monthly file:")
for col in monthly.columns:
    print(f"    {col}")

print("\n" + "=" * 60)
print("DONE")
print("=" * 60)
print("\nNext step:")

---
## Step 3: Merge Panel with Climate Covariates

Joins the monthly break panel with monthly climate covariates on `month_str`. Adds lagged climate variables at t−1, t−2, t−3 months (physically motivated: frost penetrates to pipe depth with a delay of weeks).

**Collinear covariates removed before modelling:**
- TD (thawing days): r = −1.00 with FD — perfect collinearity
- TI (thawing index): r = 0.98 with Tmean — near-perfect collinearity


In [ ]:
print("  Load climate_monthly.csv and merge with monthly_panel_primary.csv")
print("  on 'month_str' to get the final modelling dataset.")

import os
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

PANEL_FILE   = "monthly_panel_primary.csv"
CLIMATE_FILE = "climate_monthly.csv"
OUTPUT_FILE  = "modelling_dataset.csv"

# Temperature covariates to keep
CLIMATE_COLS = [
    'Tmean', 'Tmin', 'Tmax', 'ADD',
    'FI', 'FD', 'FTC', 'FI_cum',
    'TI', 'TD', 'TIG', 'TDG',
    'missing_days'
]

# Which covariates to also add as lags (t-1, t-2, t-3)
# Physically motivated: frost at pipe depth lags surface temperature
LAG_COLS = ['Tmean', 'FI', 'FTC', 'FI_cum', 'FD', 'ADD']
LAG_MONTHS = [1, 2, 3]

print("=" * 60)
print("STEP 1: Loading files")
print("=" * 60)

try:
    script_dir = os.path.dirname(os.path.abspath(__file__))
except NameError:
    script_dir = os.getcwd()

panel   = pd.read_csv(os.path.join(script_dir, PANEL_FILE))
climate = pd.read_csv(os.path.join(script_dir, CLIMATE_FILE))

print(f"  Panel   : {panel.shape}  ({PANEL_FILE})")
print(f"  Climate : {climate.shape}  ({CLIMATE_FILE})")

print("\n" + "=" * 60)
print("STEP 2: Preparing climate file")
print("=" * 60)

# Keep only the columns we need
climate_keep = ['month_str'] + CLIMATE_COLS
climate_slim = climate[climate_keep].copy()

print(f"  Climate columns kept: {len(climate_keep)}")
print(f"  {climate_keep}")

print("\n" + "=" * 60)
print("STEP 3: Merging panel with climate")
print("=" * 60)

# Panel covers 1997-2025, climate covers 1996-2025
# Left join: keep all panel rows, match climate where available
# The 1996 climate year is only needed for lag calculation (added below)
merged = panel.merge(climate_slim, on='month_str', how='left')

print(f"  Rows after merge: {len(merged)} (expect 1044)")
print(f"  Columns after merge: {len(merged.columns)}")

# Check all climate rows matched
unmatched = merged['Tmean'].isna().sum()
print(f"  Rows with no climate match: {unmatched} (expect 0)")

print("\n" + "=" * 60)
print("STEP 4: Adding lagged climate covariates")
print("=" * 60)

# To build lags correctly we need the 1996 climate data too
# because 1997-01 lag-1 = 1996-12, lag-2 = 1996-11, lag-3 = 1996-10
climate_full = climate[['month_str'] + LAG_COLS].copy()
climate_full['month_dt'] = pd.to_datetime(climate_full['month_str'] + '-01')
climate_full = climate_full.sort_values('month_dt').reset_index(drop=True)

# Build a lookup: month_str -> lagged values
lag_lookup = climate_full.set_index('month_str')[LAG_COLS]

def get_lag_month_str(month_str, lag):
    """Return the month_str that is 'lag' months before the given month_str."""
    dt = pd.to_datetime(month_str + '-01') - pd.DateOffset(months=lag)
    return dt.strftime('%Y-%m')

# Add lags to merged dataset
for col in LAG_COLS:
    for lag in LAG_MONTHS:
        new_col = f"{col}_lag{lag}"
        merged[new_col] = merged['month_str'].apply(
            lambda ms: lag_lookup.loc[get_lag_month_str(ms, lag), col]
            if get_lag_month_str(ms, lag) in lag_lookup.index else np.nan
        )
        n_null = merged[new_col].isna().sum()
        status = "OK" if n_null == 0 else f"WARNING: {n_null} nulls"
        print(f"  Added {new_col:<20} {status}")

print("\n" + "=" * 60)
print("STEP 5: Organising final column order")
print("=" * 60)

# Identity columns
id_cols = ['material', 'month_str', 'year', 'month_num']

# Target variable
target = ['break_count', 'km_primary', 'failure_rate']

# Pipe / network features
pipe_cols = ['sin_month', 'cos_month', 'time_index']

# Current-month climate
climate_current = [c for c in CLIMATE_COLS if c != 'missing_days']

# Lagged climate
lag_col_names = [f"{c}_lag{l}" for c in LAG_COLS for l in LAG_MONTHS]

# Quality flag
quality = ['missing_days']

final_cols = id_cols + target + pipe_cols + climate_current + lag_col_names + quality

# Keep only columns that exist
final_cols = [c for c in final_cols if c in merged.columns]
dataset = merged[final_cols].copy()

print(f"  Final columns: {len(dataset.columns)}")
for col in dataset.columns:
    print(f"    {col}")

print("\n" + "=" * 60)
print("STEP 6: Sanity checks")
print("=" * 60)

# Check 1: shape
print(f"\n  Shape: {dataset.shape}")
print(f"  Expected rows: 1044 (3 materials x 348 months)")

# Check 2: nulls
print(f"\n  Null counts:")
nulls = dataset.isnull().sum()
null_cols = nulls[nulls > 0]
if len(null_cols) == 0:
    print("    None — all columns complete")
else:
    print(null_cols.to_string())

# Check 3: materials and date range
print(f"\n  Materials: {dataset['material'].unique().tolist()}")
print(f"  Date range: {dataset['month_str'].min()} to {dataset['month_str'].max()}")

# Check 4: failure rate by material
print(f"\n  Mean failure rate (breaks/km/month) by material:")
print(dataset.groupby('material')['failure_rate'].mean().round(5).to_string())

# Check 5: climate signal visible
print(f"\n  Mean FI by material x season (CI should show strong winter signal):")
dataset['winter'] = dataset['month_num'].isin([12,1,2])
print(dataset.groupby(['material','winter'])['FI'].mean().round(1).to_string())
dataset.drop(columns=['winter'], inplace=True)

# Check 6: lag columns look right
print(f"\n  Spot check: Jan 1997 CI row")
jan97 = dataset[(dataset['month_str']=='1997-01')&(dataset['material']=='CI')]
print(jan97[['month_str','Tmean','Tmean_lag1','Tmean_lag2','Tmean_lag3']].to_string(index=False))
print("  (lag1 = Dec 1996, lag2 = Nov 1996, lag3 = Oct 1996 — should be cold)")

# Check 7: time_index
print(f"\n  Time index range: {dataset['time_index'].min()} to {dataset['time_index'].max()}")
print(f"  (expect 0 to 347)")

# Check 8: missing_days flag
months_flagged = dataset[dataset['missing_days']>5]['month_str'].unique()
print(f"\n  Months with >5 gap-filled climate days (flag in paper):")
print(f"    {sorted(set(months_flagged))}")

print("\n" + "=" * 60)
print("STEP 7: Saving")
print("=" * 60)

out_path = os.path.join(script_dir, OUTPUT_FILE)
dataset.to_csv(out_path, index=False)

print(f"\n  Saved: {out_path}")
print(f"  Shape: {dataset.shape}")

print("\n" + "=" * 60)
print("DONE")
print("=" * 60)
print("""
What you have now:
  modelling_dataset.csv — the final dataset ready for modelling

Columns:
  Identity  : material, month_str, year, month_num
  Target    : break_count, km_primary, failure_rate
  Network   : sin_month, cos_month, time_index
  Climate   : Tmean, Tmin, Tmax, ADD, FI, FD, FTC, FI_cum,
              TI, TD, TIG, TDG
  Lags      : Tmean_lag1/2/3, FI_lag1/2/3, FTC_lag1/2/3,
              FI_cum_lag1/2/3, FD_lag1/2/3, ADD_lag1/2/3
  Quality   : missing_days

Next step:
  Load modelling_dataset.csv and fit the negative binomial model
  separately for each material (CI, DI, PVC).
""")


---
## Step 4: Negative Binomial Count Regression Models

Fits negative binomial GLMs for CI and DI separately with log(km) exposure offset. Uses backward stepwise AIC feature selection.

**Validation protocol:**
- **Split B (primary):** Train 1997–2020, Test 2021–2025
- **Split A (robustness):** Train 1997–2016, Val 2017–2021, Test 2022–2025
- Both splits are strictly temporal — no future data leaks into training

**Selected features:**
- CI: FI, Tmean_lag1, FD_lag1, cos_month (identical in both splits → robust)
- DI Split B: FI, FI_lag2, FD_lag2 (primary — lower MAE_rate)
- DI Split A: FI, Tmean_lag2, FTC_lag1, FTC_lag3, FD_lag2

**Model performance vs Khashei et al. (2024) range 0.040–0.192 breaks/km/year:**
- CI: MAE_rate = 0.085 (Split B primary), 0.100 (Split A robustness)
- DI: MAE_rate = 0.022 (Split B primary), 0.023 (Split A robustness)


In [ ]:
import statsmodels.api as sm
from statsmodels.genmod.generalized_linear_model import GLM
from statsmodels.genmod import families
import scipy.stats as stats
warnings.filterwarnings('ignore')

DATA_FILE = "modelling_dataset.csv"

# Split A
A_TRAIN_END  = 2016
A_VAL_START  = 2017
A_VAL_END    = 2021
A_TEST_START = 2022

# Split B
B_TRAIN_END  = 2020
B_TEST_START = 2021

# Rolling CV folds for Split B model selection
# Fit on 1997-FOLD_START, predict FOLD_START+1 to FOLD_START+1
B_CV_FOLD_STARTS = list(range(2010, 2020))  # 10 folds

# Climate covariates (TD, TI dropped — collinear)
CLIMATE_CURRENT = ['Tmean', 'FI', 'FTC', 'FI_cum', 'FD', 'ADD', 'TIG', 'TDG']
LAG_COLS        = ['Tmean', 'FI', 'FTC', 'FI_cum', 'FD', 'ADD']
LAG_MONTHS      = [1, 2, 3]
CLIMATE_LAGS    = [f'{c}_lag{l}' for c in LAG_COLS for l in LAG_MONTHS]
NETWORK_COLS    = ['sin_month', 'cos_month', 'time_index']
ALL_FEATURES    = CLIMATE_CURRENT + CLIMATE_LAGS + NETWORK_COLS

try:
    script_dir = os.path.dirname(os.path.abspath(__file__))
except NameError:
    script_dir = os.getcwd()

print("=" * 60)
print("Loading data")
print("=" * 60)
df = pd.read_csv(os.path.join(script_dir, DATA_FILE))
print(f"  Shape: {df.shape}")
print(f"  Date range: {df['month_str'].min()} to {df['month_str'].max()}")

# Print split sizes
print("\n  Split sizes:")
for mat in ['CI', 'DI']:
    m = df[df['material'] == mat]
    trA = m[m['year'] <= A_TRAIN_END]
    vaA = m[(m['year'] >= A_VAL_START) & (m['year'] <= A_VAL_END)]
    teA = m[m['year'] >= A_TEST_START]
    trB = m[m['year'] <= B_TRAIN_END]
    teB = m[m['year'] >= B_TEST_START]
    print(f"  {mat} Split A: train={len(trA)}({trA.break_count.sum()}br) "
          f"val={len(vaA)}({vaA.break_count.sum()}br) "
          f"test={len(teA)}({teA.break_count.sum()}br)")
    print(f"  {mat} Split B: train={len(trB)}({trB.break_count.sum()}br) "
          f"test={len(teB)}({teB.break_count.sum()}br)")

def fit_negbinom(train_df, features, offset_col='km_primary'):
    X      = sm.add_constant(train_df[features])
    y      = train_df['break_count']
    offset = np.log(train_df[offset_col].clip(lower=0.001))
    model  = GLM(y, X, family=families.NegativeBinomial(alpha=1.0), offset=offset)
    return model.fit(maxiter=200, disp=False)


def predict_negbinom(result, pred_df, features,
                     offset_col='km_primary', alpha=0.10):
    X      = sm.add_constant(pred_df[features], has_constant='add')
    offset = np.log(pred_df[offset_col].clip(lower=0.001))
    mu     = result.predict(X, offset=offset)
    nb_a   = result.scale
    lower  = stats.nbinom.ppf(alpha/2,   n=1/nb_a, p=1/(1 + nb_a*mu))
    upper  = stats.nbinom.ppf(1-alpha/2, n=1/nb_a, p=1/(1 + nb_a*mu))
    return mu.values, lower, upper


def monthly_metrics(actual, predicted, lower, upper, baseline_mae):
    mae     = float(np.mean(np.abs(actual - predicted)))
    mase    = mae / baseline_mae if baseline_mae > 0 else np.nan
    cov90   = float(np.mean((actual >= lower) & (actual <= upper))) * 100
    bias    = float(np.mean(predicted - actual))
    return {'MAE_monthly': round(mae,3), 'MASE': round(mase,3),
            'Coverage_90': round(cov90,1), 'Bias': round(bias,3)}


def annual_failure_rate_mae(pred_df):
    """
    Convert monthly predictions to annual failure rate and compute MAE.
    failure_rate = annual_breaks / km_in_service
    This is comparable to Khashei et al. (2024).
    """
    ann = pred_df.groupby('year').agg(
        actual_breaks    = ('break_count', 'sum'),
        predicted_breaks = ('predicted',   'sum'),
        km               = ('km_primary',  'mean')
    ).reset_index()
    ann['actual_rate']    = ann['actual_breaks']    / ann['km']
    ann['predicted_rate'] = ann['predicted_breaks'] / ann['km']
    ann['abs_error_rate'] = np.abs(ann['actual_rate'] - ann['predicted_rate'])
    mae_rate = float(ann['abs_error_rate'].mean())
    return round(mae_rate, 4), ann


def annual_count_mae(actual, predicted, years):
    """Sum monthly to annual and compute MAE on counts."""
    temp = pd.DataFrame({'a': actual, 'p': predicted, 'y': years})
    ann  = temp.groupby('y').sum()
    return round(float(np.mean(np.abs(ann['a'] - ann['p']))), 2)


def stepwise_aic(train_df, features, offset_col='km_primary', verbose=False):
    """Backward stepwise AIC selection."""
    current = features.copy()
    try:
        current_aic = fit_negbinom(train_df, current, offset_col).aic
    except Exception:
        return features
    improved = True
    while improved and len(current) > 1:
        improved = False
        best_aic, best_drop = current_aic, None
        for feat in current:
            trial = [f for f in current if f != feat]
            try:
                aic = fit_negbinom(train_df, trial, offset_col).aic
                if aic < best_aic:
                    best_aic, best_drop = aic, feat
            except Exception:
                pass
        if best_drop:
            current.remove(best_drop)
            current_aic = best_aic
            improved = True
            if verbose:
                print(f"      drop '{best_drop}' → AIC={best_aic:.1f}")
    return current


def rolling_cv_select(mat_df, train_end, fold_starts,
                       offset_col='km_primary', verbose=False):
    """
    Rolling-origin CV to select features for Split B.
    For each fold year F: train on all data up to F, predict F+1.
    Returns the feature set with lowest mean CV MAE.
    """
    print(f"    Rolling-origin CV with {len(fold_starts)} folds "
          f"({fold_starts[0]}-{fold_starts[-1]})...")

    # Step 1: select features on full training block using AIC
    full_train = mat_df[mat_df['year'] <= train_end]
    selected   = stepwise_aic(full_train, ALL_FEATURES, verbose=False)
    print(f"    AIC-selected features ({len(selected)}): {selected}")

    # Step 2: estimate CV MAE for selected vs full feature set
    cv_mae_selected, cv_mae_full = [], []

    for fold_start in fold_starts:
        cv_train = mat_df[mat_df['year'] <= fold_start]
        cv_val   = mat_df[mat_df['year'] == fold_start + 1]
        if len(cv_val) == 0 or cv_val['break_count'].sum() == 0:
            continue
        try:
            # Selected features
            r_sel  = fit_negbinom(cv_train, selected, offset_col)
            p_sel, _, _ = predict_negbinom(r_sel, cv_val, selected, offset_col)
            cv_mae_selected.append(np.mean(np.abs(cv_val['break_count'].values - p_sel)))

            # Full features
            r_full = fit_negbinom(cv_train, ALL_FEATURES, offset_col)
            p_full, _, _ = predict_negbinom(r_full, cv_val, ALL_FEATURES, offset_col)
            cv_mae_full.append(np.mean(np.abs(cv_val['break_count'].values - p_full)))
        except Exception:
            pass

    mae_sel  = np.mean(cv_mae_selected)  if cv_mae_selected  else np.nan
    mae_full = np.mean(cv_mae_full) if cv_mae_full else np.nan
    print(f"    CV MAE — selected: {mae_sel:.3f}  full: {mae_full:.3f}")

    best = selected if mae_sel <= mae_full else ALL_FEATURES
    print(f"    → Using {'selected' if best is selected else 'full'} features for Split B")
    return best

print("\n" + "=" * 60)
print("M0 — Seasonal naive baseline")
print("=" * 60)

m0 = {}
for mat in ['CI', 'DI', 'PVC']:
    mat_df = df[df['material'] == mat]

    # Split A baseline
    trA = mat_df[mat_df['year'] <= A_TRAIN_END]
    vaA = mat_df[(mat_df['year'] >= A_VAL_START) & (mat_df['year'] <= A_VAL_END)]
    teA = mat_df[mat_df['year'] >= A_TEST_START]
    means_A = trA.groupby('month_num')['break_count'].mean()

    # Split B baseline
    trB = mat_df[mat_df['year'] <= B_TRAIN_END]
    teB = mat_df[mat_df['year'] >= B_TEST_START]
    means_B = trB.groupby('month_num')['break_count'].mean()

    m0[mat] = {
        'means_A': means_A, 'means_B': means_B,
        'val_A_mae':  float(np.mean(np.abs(vaA['break_count'].values - vaA['month_num'].map(means_A).values))),
        'test_A_mae': float(np.mean(np.abs(teA['break_count'].values - teA['month_num'].map(means_A).values))),
        'test_B_mae': float(np.mean(np.abs(teB['break_count'].values - teB['month_num'].map(means_B).values))),
        'teA': teA, 'teB': teB, 'vaA': vaA,
    }
    print(f"  {mat}: val_A_MAE={m0[mat]['val_A_mae']:.3f}  "
          f"test_A_MAE={m0[mat]['test_A_mae']:.3f}  "
          f"test_B_MAE={m0[mat]['test_B_mae']:.3f}")

all_results  = []
all_preds    = {}
all_coefs    = {}

for mat in ['CI', 'DI']:
    mat_df = df[df['material'] == mat].copy()

    print(f"\n{'='*60}")
    print(f"Material: {mat}")
    print('='*60)

    # ── SPLIT A ───────────────────────────────────────────────────────────────
    print("\n  --- Split A (train 1997-2016, val 2017-2021, test 2022-2025) ---")

    trA = mat_df[mat_df['year'] <= A_TRAIN_END]
    vaA = mat_df[(mat_df['year'] >= A_VAL_START) & (mat_df['year'] <= A_VAL_END)]
    teA = mat_df[mat_df['year'] >= A_TEST_START]

    base_val_A  = m0[mat]['val_A_mae']
    base_test_A = m0[mat]['test_A_mae']

    # M1 Split A
    try:
        m1A = fit_negbinom(trA, ALL_FEATURES)
        p_v, lo_v, hi_v = predict_negbinom(m1A, vaA, ALL_FEATURES)
        met_m1A_val = monthly_metrics(vaA['break_count'].values, p_v, lo_v, hi_v, base_val_A)
        met_m1A_val['MAE_annual'] = annual_count_mae(vaA['break_count'].values, p_v, vaA['year'].values)
        print(f"  M1 val:  MAE_mo={met_m1A_val['MAE_monthly']}  "
              f"MAE_ann={met_m1A_val['MAE_annual']}  MASE={met_m1A_val['MASE']}")
    except Exception as e:
        print(f"  M1 Split A failed: {e}")
        m1A = None

    # M2 Split A (stepwise on training data)
    print(f"  Stepwise AIC selection (Split A)...")
    selA = stepwise_aic(trA, ALL_FEATURES, verbose=True)
    print(f"  Selected {len(selA)} features: {selA}")

    try:
        m2A = fit_negbinom(trA, selA)
        p_v, lo_v, hi_v = predict_negbinom(m2A, vaA, selA)
        met_m2A_val = monthly_metrics(vaA['break_count'].values, p_v, lo_v, hi_v, base_val_A)
        met_m2A_val['MAE_annual'] = annual_count_mae(vaA['break_count'].values, p_v, vaA['year'].values)
        print(f"  M2 val:  MAE_mo={met_m2A_val['MAE_monthly']}  "
              f"MAE_ann={met_m2A_val['MAE_annual']}  MASE={met_m2A_val['MASE']}")

        # Choose best Split A model
        best_A_features = selA if met_m2A_val['MASE'] <= met_m1A_val['MASE'] else ALL_FEATURES
        best_A_name     = 'M2' if met_m2A_val['MASE'] <= met_m1A_val['MASE'] else 'M1'

        # Refit on train+val, evaluate test
        trva_A = mat_df[mat_df['year'] <= A_VAL_END]
        best_A_refit = fit_negbinom(trva_A, best_A_features)
        p_t, lo_t, hi_t = predict_negbinom(best_A_refit, teA, best_A_features)

        met_A_test = monthly_metrics(teA['break_count'].values, p_t, lo_t, hi_t, base_test_A)
        met_A_test['MAE_annual'] = annual_count_mae(teA['break_count'].values, p_t, teA['year'].values)

        # Rate-based MAE
        pred_A_df = teA.copy()
        pred_A_df['predicted'] = p_t
        mae_rate_A, ann_A = annual_failure_rate_mae(pred_A_df)
        met_A_test['MAE_rate'] = mae_rate_A

        print(f"\n  Split A TEST ({best_A_name}): "
              f"MAE_mo={met_A_test['MAE_monthly']}  "
              f"MAE_ann={met_A_test['MAE_annual']}  "
              f"MAE_rate={mae_rate_A}  "
              f"MASE={met_A_test['MASE']}  "
              f"Cov90={met_A_test['Coverage_90']}%  "
              f"Bias={met_A_test['Bias']}")

        print(f"  Annual failure rates (Split A test):")
        print(ann_A[['year','actual_breaks','predicted_breaks',
                      'actual_rate','predicted_rate']].round(4).to_string(index=False))

        # Store
        pred_A_df['lower_90'] = lo_t
        pred_A_df['upper_90'] = hi_t
        pred_A_df['residual'] = teA['break_count'].values - p_t
        all_preds[f'{mat}_A'] = pred_A_df

        coef_A = pd.DataFrame({
            'feature': best_A_refit.params.index,
            'coefficient': best_A_refit.params.values,
            'std_err': best_A_refit.bse.values,
            'p_value': best_A_refit.pvalues.values,
            'significant': best_A_refit.pvalues.values < 0.05
        })
        all_coefs[f'{mat}_A'] = coef_A

        all_results.append({
            'material': mat, 'split': 'A', 'model': best_A_name,
            'train_years': '1997-2016', 'test_years': '2022-2025',
            **met_A_test
        })

    except Exception as e:
        print(f"  M2/test Split A failed: {e}")

# ── SPLIT B ───────────────────────────────────────────────────────────────
    print(f"\n  --- Split B (train 1997-2020, test 2021-2025) ---")

    trB = mat_df[mat_df['year'] <= B_TRAIN_END]
    teB = mat_df[mat_df['year'] >= B_TEST_START]
    base_test_B = m0[mat]['test_B_mae']

    # Feature selection via rolling CV
    selB = rolling_cv_select(mat_df, B_TRAIN_END, B_CV_FOLD_STARTS)

    try:
        m2B = fit_negbinom(trB, selB)
        p_t, lo_t, hi_t = predict_negbinom(m2B, teB, selB)

        met_B_test = monthly_metrics(teB['break_count'].values, p_t, lo_t, hi_t, base_test_B)
        met_B_test['MAE_annual'] = annual_count_mae(teB['break_count'].values, p_t, teB['year'].values)

        # Rate-based MAE
        pred_B_df = teB.copy()
        pred_B_df['predicted'] = p_t
        mae_rate_B, ann_B = annual_failure_rate_mae(pred_B_df)
        met_B_test['MAE_rate'] = mae_rate_B

        print(f"\n  Split B TEST: "
              f"MAE_mo={met_B_test['MAE_monthly']}  "
              f"MAE_ann={met_B_test['MAE_annual']}  "
              f"MAE_rate={mae_rate_B}  "
              f"MASE={met_B_test['MASE']}  "
              f"Cov90={met_B_test['Coverage_90']}%  "
              f"Bias={met_B_test['Bias']}")

        print(f"  Annual failure rates (Split B test):")
        print(ann_B[['year','actual_breaks','predicted_breaks',
                      'actual_rate','predicted_rate']].round(4).to_string(index=False))

        # Store
        pred_B_df['lower_90'] = lo_t
        pred_B_df['upper_90'] = hi_t
        pred_B_df['residual'] = teB['break_count'].values - p_t
        all_preds[f'{mat}_B'] = pred_B_df

        coef_B = pd.DataFrame({
            'feature': m2B.params.index,
            'coefficient': m2B.params.values,
            'std_err': m2B.bse.values,
            'p_value': m2B.pvalues.values,
            'significant': m2B.pvalues.values < 0.05
        })
        all_coefs[f'{mat}_B'] = coef_B

        all_results.append({
            'material': mat, 'split': 'B', 'model': 'M2_rollingCV',
            'train_years': '1997-2020', 'test_years': '2021-2025',
            **met_B_test
        })

    except Exception as e:
        print(f"  Split B failed: {e}")

print(f"\n{'='*60}")
print("PVC — Seasonal baseline only (insufficient test events)")
print('='*60)
pvc = df[df['material']=='PVC']
teA_pvc = pvc[pvc['year'] >= A_TEST_START]
teB_pvc = pvc[pvc['year'] >= B_TEST_START]
print(f"  Split A test breaks: {teA_pvc['break_count'].sum()} — too few for climate model")
print(f"  Split B test breaks: {teB_pvc['break_count'].sum()} — too few for climate model")

print(f"\n{'='*60}")
print("RESULTS SUMMARY")
print('='*60)
results_df = pd.DataFrame(all_results)
print(results_df[['material','split','model','train_years','test_years',
                   'MAE_monthly','MAE_annual','MAE_rate','MASE',
                   'Coverage_90','Bias']].to_string(index=False))

print(f"\n  MAE_rate = breaks/km/year (comparable to Khashei et al. 2024: 0.040-0.192)")
print(f"  MASE < 1 = beats seasonal naive baseline")
print(f"  If Split B MAE < Split A MAE: more training data helped")
print(f"  If similar: model is stable regardless of split choice")

print(f"\n{'='*60}")
print("Saving outputs")
print('='*60)

results_df.to_csv(os.path.join(script_dir, 'model_results_both_splits.csv'), index=False)
print("  Saved: model_results_both_splits.csv")

for key, coef_df in all_coefs.items():
    mat, split = key.split('_')
    fname = f'coefficients_{mat.lower()}_split{split}.csv'
    coef_df.to_csv(os.path.join(script_dir, fname), index=False)
    print(f"  Saved: {fname}")

for key, pred_df in all_preds.items():
    mat, split = key.split('_')
    fname = f'predictions_{mat.lower()}_split{split}.csv'
    pred_df[['material','month_str','year','month_num','break_count',
             'km_primary','predicted','lower_90','upper_90','residual']
            ].to_csv(os.path.join(script_dir, fname), index=False)
    print(f"  Saved: {fname}")

print("\n" + "="*60)
print("DONE")
print("="*60)


---
## Step 5: Process CMIP6 NetCDF Files

Reads 24 CMIP6 NetCDF files (2 GCMs × 4 SSPs × 3 variables) from PCIC CanDCS-M6 (multivariate bias-corrected), calculates the same monthly climate covariates as the historical data, and saves one CSV per model-scenario combination.

**Climate data:** CanDCS-M6 (MBCn multivariate bias correction)  
**GCMs:** CanESM5, MIROC6  
**Scenarios:** SSP1-2.6, SSP2-4.5, SSP3-7.0, SSP5-8.5  
**Period:** 2026–2100  
**Location:** Kitchener (43.46°N, −80.54°W)

**Note on FTC bias:** CanESM5 projects 13–14 freeze-thaw cycles/month in January versus observed 8.5. This is a known limitation of threshold-based variables in GCM output. Stated as a study limitation.


In [ ]:
!pip install netCDF4

import os
import glob
import numpy as np
import pandas as pd
import netCDF4 as nc
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Projection period — historical overlap excluded
PROJ_START_YEAR = 2026
PROJ_END_YEAR   = 2100

# Historical overlap period — used for bias check
HIST_START_YEAR = 1995
HIST_END_YEAR   = 2014

MISSING_VALUE = 32767.0

try:
    script_dir = os.path.dirname(os.path.abspath(__file__))
except NameError:
    script_dir = os.getcwd()

print("=" * 60)
print("STEP 1: Finding NC files")
print("=" * 60)

nc_files = glob.glob(os.path.join(script_dir, "*.nc"))

if not nc_files:
    raise FileNotFoundError(
        "No .nc files found in current folder.\n"
        "Put all CMIP6 .nc files in the same folder as this script."
    )

print(f"  Found {len(nc_files)} NC files:")
for f in sorted(nc_files):
    print(f"    {os.path.basename(f)}")

print("\n" + "=" * 60)
print("STEP 2: Grouping files by model and scenario")
print("=" * 60)

# File naming pattern (underscores replacing + and spaces):
# tasmax_day_MBCn_PCIC-Blend_MIROC6_historical_ssp370_r1i1p1f1_gn_...nc
# We need to extract: variable (tasmax/tasmin/pr), model (CanESM5/MIROC6),
# scenario (ssp126/ssp245/ssp370/ssp585)

groups = {}  # key = (model, scenario), value = {var: filepath}

for f in nc_files:
    base = os.path.basename(f).lower()

    # Variable
    if base.startswith('tasmax'):
        var = 'tasmax'
    elif base.startswith('tasmin'):
        var = 'tasmin'
    elif base.startswith('pr'):
        var = 'pr'
    else:
        print(f"  WARNING: Unknown variable in {base}, skipping")
        continue

    # Model
    if 'canesm5' in base:
        model = 'CanESM5'
    elif 'miroc6' in base:
        model = 'MIROC6'
    else:
        print(f"  WARNING: Unknown model in {base}, skipping")
        continue

    # Scenario
    for ssp in ['ssp126', 'ssp245', 'ssp370', 'ssp585']:
        if ssp in base:
            scenario = ssp
            break
    else:
        print(f"  WARNING: Unknown scenario in {base}, skipping")
        continue

    key = (model, scenario)
    if key not in groups:
        groups[key] = {}
    groups[key][var] = f

print(f"  Groups found: {len(groups)}")
for (model, scenario), vars_dict in sorted(groups.items()):
    vars_present = sorted(vars_dict.keys())
    status = "✓ complete" if len(vars_present) == 3 else f"INCOMPLETE: {vars_present}"
    print(f"    {model} {scenario}: {status}")

def read_nc_timeseries(filepath, varname):
    """
    Read a PCIC CanDCS-M6 NetCDF file.
    Returns a DataFrame with columns: date, <varname>
    Handles missing values (32767) and flattens the (time,1,1) shape.
    Auto-detects the actual variable name in the file in case it differs.
    """
    ds = nc.Dataset(filepath)

    # Auto-detect the climate variable (not time/lat/lon)
    coord_vars = {'time', 'lat', 'lon', 'latitude', 'longitude'}
    data_vars = [v for v in ds.variables if v not in coord_vars]
    actual_varname = data_vars[0] if data_vars else varname

    # Time: days since 1950-01-01
    time_var  = ds.variables['time']
    time_vals = time_var[:].data
    base_date = datetime(1950, 1, 1)
    dates = [base_date + timedelta(days=float(int(t))) for t in time_vals]

    # Variable data — shape (time, 1, 1)
    data = ds.variables[actual_varname][:].data.flatten()

    # Mask missing values
    data = np.where(np.abs(data - MISSING_VALUE) < 1, np.nan, data)

    ds.close()

    # Always return with the requested varname as column
    df = pd.DataFrame({'date': dates, varname: data})
    df['date'] = pd.to_datetime(df['date'])
    return df

def calc_gradients(tmean_series):
    """TIG and TDG: max rate of temperature change between consecutive days."""
    t = tmean_series.dropna().values
    if len(t) < 2:
        return np.nan, np.nan
    diffs = np.diff(t)
    return float(np.max(diffs)), float(np.max(-diffs))


def calc_fi_cumulative(daily_df, year, month):
    """
    Cumulative freezing index since start of freeze season (Oct previous year).
    """
    if month < 10:
        season_start = pd.Timestamp(year=year-1, month=10, day=1)
    else:
        season_start = pd.Timestamp(year=year, month=10, day=1)
    season_end = pd.Timestamp(year=year, month=month, day=1) + pd.offsets.MonthEnd(0)
    mask = (daily_df['date'] >= season_start) & (daily_df['date'] <= season_end)
    fi_cum = daily_df.loc[mask, 'tasmin'].apply(
        lambda x: min(x, 0) if pd.notna(x) else 0
    ).sum()
    return round(fi_cum, 2)


def build_monthly_covariates(daily_df, year_start, year_end):
    """
    Given a daily DataFrame with columns: date, tasmax, tasmin, pr
    Returns monthly covariate DataFrame for year_start to year_end.
    """
    # Filter to requested period
    df = daily_df[
        (daily_df['date'].dt.year >= year_start) &
        (daily_df['date'].dt.year <= year_end)
    ].copy()

    df['year']  = df['date'].dt.year
    df['month'] = df['date'].dt.month

    # Tmean approximation from tasmax and tasmin
    # PCIC files provide tasmax and tasmin but not tmean directly
    # Standard approximation: Tmean = (Tmax + Tmin) / 2
    df['Tmean'] = (df['tasmax'] + df['tasmin']) / 2

    monthly_rows = []

    for (yr, mo), grp in df.groupby(['year', 'month']):
        grp = grp.copy().reset_index(drop=True)

        # Temperature level
        Tmean = grp['Tmean'].mean()
        Tmin  = grp['tasmin'].mean()
        Tmax  = grp['tasmax'].mean()
        ADD   = (grp['tasmax'] - grp['tasmin']).mean()

        # Frost
        FI  = grp['Tmean'].apply(lambda x: min(x,0) if pd.notna(x) else 0).sum()
        FD  = int((grp['Tmean'] < 0).sum())
        FTC = int(((grp['tasmin'] < 0) & (grp['tasmax'] > 0)).sum())

        # Thaw
        TI  = grp['Tmean'].apply(lambda x: max(x,0) if pd.notna(x) else 0).sum()
        TD  = int((grp['Tmean'] > 0).sum())

        # Gradients
        TIG, TDG = calc_gradients(grp['Tmean'])

        # Cumulative FI since freeze season start
        FI_cum = calc_fi_cumulative(df, yr, mo)

        monthly_rows.append({
            'year':     yr,
            'month':    mo,
            'month_str': f"{yr}-{mo:02d}",
            'Tmean':    round(Tmean, 3) if pd.notna(Tmean) else np.nan,
            'Tmin':     round(Tmin,  3) if pd.notna(Tmin)  else np.nan,
            'Tmax':     round(Tmax,  3) if pd.notna(Tmax)  else np.nan,
            'ADD':      round(ADD,   3) if pd.notna(ADD)    else np.nan,
            'FI':       round(FI,    2),
            'FD':       FD,
            'FTC':      FTC,
            'FI_cum':   FI_cum,
            'TI':       round(TI, 2),
            'TD':       TD,
            'TIG':      round(TIG, 3) if pd.notna(TIG) else np.nan,
            'TDG':      round(TDG, 3) if pd.notna(TDG) else np.nan,
        })

    return pd.DataFrame(monthly_rows).sort_values(['year','month']).reset_index(drop=True)

print("\n" + "=" * 60)
print("STEP 3: Processing each model-scenario combination")
print("=" * 60)

all_summaries = []

for (model, scenario), vars_dict in sorted(groups.items()):

    print(f"\n  --- {model} {scenario} ---")

    # Check all three variables present
    missing_vars = [v for v in ['tasmax','tasmin','pr'] if v not in vars_dict]
    if missing_vars:
        print(f"  SKIPPING — missing variables: {missing_vars}")
        continue

    # Read all three variables
    print(f"  Reading tasmax...")
    df_max  = read_nc_timeseries(vars_dict['tasmax'], 'tasmax')
    print(f"  Reading tasmin...")
    df_min  = read_nc_timeseries(vars_dict['tasmin'], 'tasmin')
    print(f"  Reading pr...")
    df_pr   = read_nc_timeseries(vars_dict['pr'], 'pr')

    # Merge on date
    daily = df_max.merge(df_min, on='date').merge(df_pr, on='date')
    print(f"  Daily rows: {len(daily)} | date range: {daily['date'].min().date()} to {daily['date'].max().date()}")

    # Check for missing values
    for col in ['tasmax','tasmin','pr']:
        n_miss = daily[col].isna().sum()
        if n_miss > 0:
            print(f"  WARNING: {col} has {n_miss} missing values")

    # Build monthly covariates for projection period
    print(f"  Calculating monthly covariates ({PROJ_START_YEAR}-{PROJ_END_YEAR})...")
    monthly_proj = build_monthly_covariates(daily, PROJ_START_YEAR, PROJ_END_YEAR)
    monthly_proj['model']    = model
    monthly_proj['scenario'] = scenario

    expected_months = (PROJ_END_YEAR - PROJ_START_YEAR + 1) * 12
    print(f"  Monthly rows: {len(monthly_proj)} (expect {expected_months})")

    # Build monthly covariates for historical overlap (for bias check)
    # Note: some files may start after HIST_END_YEAR — handle gracefully
    daily_min_yr = daily['date'].dt.year.min()
    if daily_min_yr <= HIST_END_YEAR:
        monthly_hist = build_monthly_covariates(daily, HIST_START_YEAR, HIST_END_YEAR)
        monthly_hist['model']    = model
        monthly_hist['scenario'] = scenario
        out_hist = os.path.join(script_dir, f"cmip6_hist_{model}_{scenario}.csv")
        monthly_hist.to_csv(out_hist, index=False)
    else:
        print(f"  Note: file starts in {daily_min_yr}, no historical overlap available for bias check")

    # Quick sanity check on projection data
    jan_fi = monthly_proj[monthly_proj['month']==1]['FI'].mean()
    jul_tmp = monthly_proj[monthly_proj['month']==7]['Tmean'].mean()
    print(f"  Sanity: Jan mean FI={jan_fi:.1f} (expect negative), Jul Tmean={jul_tmp:.1f} (expect warm)")

    # Save projection file
    out_proj = os.path.join(script_dir, f"cmip6_monthly_{model}_{scenario}.csv")
    monthly_proj.to_csv(out_proj, index=False)
    print(f"  Saved: {os.path.basename(out_proj)}")

    # (historical overlap file saved inside the if-block above)

    # Summary stats
    all_summaries.append({
        'model': model,
        'scenario': scenario,
        'proj_months': len(monthly_proj),
        'jan_fi_mean': round(jan_fi, 1),
        'jul_tmean': round(jul_tmp, 1),
        'jan_ftc_mean': round(monthly_proj[monthly_proj['month']==1]['FTC'].mean(), 1),
    })

print("\n" + "=" * 60)
print("STEP 4: Bias check — GCM historical vs observed (1995-2014)")
print("=" * 60)
print("  Comparing GCM historical run against ECCC observed climate.")
print("  Large bias means the model systematically over/underpredicts.")
print("  For FTC especially — threshold crossings are sensitive to bias.\n")

# Load historical observed monthly data
obs_path = os.path.join(script_dir, 'climate_monthly.csv')
if os.path.exists(obs_path):
    obs = pd.read_csv(obs_path)
    obs_hist = obs[(obs['year']>=HIST_START_YEAR) & (obs['year']<=HIST_END_YEAR)]
    obs_jan = obs_hist[obs_hist['month']==1]

    print(f"  Observed (ECCC) 1995-2014:")
    print(f"    Jan Tmean mean: {obs_jan['Tmean'].mean():.1f}°C")
    print(f"    Jan FI mean:    {obs_jan['FI'].mean():.1f} deg-C-days")
    print(f"    Jan FTC mean:   {obs_jan['FTC'].mean():.1f} cycles")
    print()

    # Compare each GCM
    hist_files = glob.glob(os.path.join(script_dir, "cmip6_hist_*.csv"))
    for hf in sorted(hist_files):
        gcm = pd.read_csv(hf)
        gcm_jan = gcm[gcm['month']==1]
        model   = gcm['model'].iloc[0]
        scenario= gcm['scenario'].iloc[0]
        bias_tmean = gcm_jan['Tmean'].mean() - obs_jan['Tmean'].mean()
        bias_fi    = gcm_jan['FI'].mean()    - obs_jan['FI'].mean()
        bias_ftc   = gcm_jan['FTC'].mean()   - obs_jan['FTC'].mean()
        print(f"  {model} {scenario} January bias:")
        print(f"    Tmean: {bias_tmean:+.2f}°C  FI: {bias_fi:+.1f}  FTC: {bias_ftc:+.1f}")
        flag = " ← FLAG" if abs(bias_ftc) > 3 else ""
        print(f"    FTC bias{flag}")
else:
    print("  climate_monthly.csv not found — skipping bias check.")
    print("  Put climate_monthly.csv in the same folder to enable this check.")

print("\n" + "=" * 60)
print("STEP 5: Summary")
print("=" * 60)

summary_df = pd.DataFrame(all_summaries)
if len(summary_df) > 0:
    print(summary_df.to_string(index=False))

print("\n  Output files:")
out_files = glob.glob(os.path.join(script_dir, "cmip6_monthly_*.csv"))
for f in sorted(out_files):
    df = pd.read_csv(f)
    print(f"    {os.path.basename(f)}: {len(df)} rows")

print("\n" + "=" * 60)
print("DONE")
print("=" * 60)
print("""
Next steps:
  1. Check the bias report above — if FTC bias > 3 cycles/month, flag in paper
  2. Load the cmip6_monthly_*.csv files in the projection script
  3. Add lags to projected covariates (same as modelling_dataset.csv)
  4. Feed into the fitted negative binomial model to get projected break rates
  5. Run attribution scenarios A, B, C, D
""")

import os

---
## Step 6: Generate Climate Projections to 2100

Projects CI and DI failure rates from 2026 to 2100 under all 8 scenarios using the fitted negative binomial models with CMIP6 climate covariates.

**Key assumption:** Network exposure frozen at 2025 km values (no replacement or expansion modelled). This isolates the climate effect on a static network.

**Main finding:** Under all 8 scenarios, both CI and DI failure rates decline through 2100:
- CI: −11% to −37% by the 2080s (higher emissions → larger decline)
- DI: −12% to −23%

This counter-intuitive result is physically correct: CI breaks are frost-driven (60% in Dec–Feb), and warming reduces frost accumulation. Consistent with Fan et al. (2023) and Bruaset & Sægrov (2018).


In [ ]:
import glob
import warnings
import numpy as np
import pandas as pd
import statsmodels.api as sm
from statsmodels.genmod.generalized_linear_model import GLM
from statsmodels.genmod import families
import scipy.stats as stats
warnings.filterwarnings('ignore')

try:
    script_dir = os.path.dirname(os.path.abspath(__file__))
except NameError:
    script_dir = os.getcwd()

# Frozen network exposure (2025 km values)
KM_CI = 151.28
KM_DI = 322.05

# Features from stepwise AIC selection (confirmed from notebook output)
CI_FEATURES  = ['FI', 'Tmean_lag1', 'FD_lag1', 'cos_month']
DI_FEATURES_A = ['FI', 'Tmean_lag2', 'FTC_lag1', 'FTC_lag3', 'FD_lag2']
DI_FEATURES_B = ['FI', 'FI_lag2', 'FD_lag2']  # primary DI model (lower MAE_rate)

# Lag columns needed to compute (lag-1, lag-2, lag-3)
LAG_COLS = ['Tmean', 'FI', 'FTC', 'FI_cum', 'FD', 'ADD']

# 30-year projection windows for summary
WINDOWS = [
    ('2026-2055', 2026, 2055),
    ('2041-2070', 2041, 2070),
    ('2071-2100', 2071, 2100),
]

# Historical baseline window (for % change calculation)
HIST_START = 1997
HIST_END   = 2025

print("=" * 60)
print("STEP 1: Fitting models on full historical data (1997-2025)")
print("=" * 60)
print("  Using same features as selected in notebook.")
print("  CI trained on 1997-2021 (train+val). DI Split B on 1997-2020.\n")

ds_path = os.path.join(script_dir, 'modelling_dataset.csv')
if not os.path.exists(ds_path):
    raise FileNotFoundError(
        "modelling_dataset.csv not found. "
        "Run build_final_dataset.py first."
    )

ds = pd.read_csv(ds_path)

def fit_nb(df, features, offset_col='km_primary'):
    X      = sm.add_constant(df[features])
    y      = df['break_count']
    offset = np.log(df[offset_col].clip(lower=0.001))
    return GLM(
        y, X,
        family=families.NegativeBinomial(alpha=1.0),
        offset=offset
    ).fit(maxiter=200, disp=False)


# CI: train on 1997-2021 (train + val, as done for final test evaluation)
ci_data    = ds[(ds['material']=='CI') & (ds['year']<=2021)].copy()
model_ci   = fit_nb(ci_data, CI_FEATURES)

# DI Split B: train on 1997-2020 (primary model with lower MAE_rate=0.0217)
di_data_B  = ds[(ds['material']=='DI') & (ds['year']<=2020)].copy()
model_di   = fit_nb(di_data_B, DI_FEATURES_B)

# DI Split A: also fit for comparison
di_data_A  = ds[(ds['material']=='DI') & (ds['year']<=2021)].copy()
model_di_A = fit_nb(di_data_A, DI_FEATURES_A)

print("  CI model coefficients:")
for feat, coef, pval in zip(model_ci.params.index,
                             model_ci.params.values,
                             model_ci.pvalues.values):
    sig = "✓" if pval < 0.05 else " "
    print(f"    {sig} {feat:20s}  coef={coef:+.5f}  p={pval:.4f}")

print("\n  DI model (Split B) coefficients:")
for feat, coef, pval in zip(model_di.params.index,
                              model_di.params.values,
                              model_di.pvalues.values):
    sig = "✓" if pval < 0.05 else " "
    print(f"    {sig} {feat:20s}  coef={coef:+.5f}  p={pval:.4f}")

print()

# Historical mean failure rate (baseline for % change)
ci_hist = ds[(ds['material']=='CI') & (ds['year']>=HIST_START) & (ds['year']<=HIST_END)]
di_hist = ds[(ds['material']=='DI') & (ds['year']>=HIST_START) & (ds['year']<=HIST_END)]

hist_ci_rate = (ci_hist.groupby('year')['break_count'].sum() / KM_CI).mean()
hist_di_rate = (di_hist.groupby('year')['break_count'].sum() / KM_DI).mean()

print(f"  Historical mean failure rate (1997-2025):")
print(f"    CI: {hist_ci_rate:.4f} breaks/km/year")
print(f"    DI: {hist_di_rate:.4f} breaks/km/year")

def add_lags(monthly_df, hist_climate_df, lag_cols=LAG_COLS, n_lags=3):
    """
    Add lag-1, lag-2, lag-3 columns to projected monthly data.
    For the first few projection months, lags come from historical data.

    monthly_df    : projected monthly covariates (2026-2100)
    hist_climate_df: historical monthly covariates (from climate_monthly.csv)
                    needed to fill lags for 2026-01 to 2026-03
    """
    # Combine historical + projected into one time series for lag computation
    hist_tail = hist_climate_df[
        hist_climate_df['year'] >= 2024
    ][['month_str','year','month'] + lag_cols].copy()

    proj = monthly_df[['month_str','year','month'] + lag_cols].copy()
    combined = pd.concat([hist_tail, proj], ignore_index=True)
    combined = combined.sort_values(['year','month']).reset_index(drop=True)
    combined['month_dt'] = pd.to_datetime(combined['month_str'] + '-01')

    # Build lookup
    lookup = combined.set_index('month_str')[lag_cols]

    def lag_str(ms, lag):
        dt = pd.to_datetime(ms + '-01') - pd.DateOffset(months=lag)
        return dt.strftime('%Y-%m')

    result = monthly_df.copy()
    for col in lag_cols:
        for lag in [1, 2, 3]:
            new_col = f"{col}_lag{lag}"
            result[new_col] = result['month_str'].apply(
                lambda ms: lookup.loc[lag_str(ms, lag), col]
                if lag_str(ms, lag) in lookup.index else np.nan
            )

    return result


def predict_nb_monthly(model, features, monthly_df, km, alpha=0.10):
    """
    Generate monthly point predictions and 90% prediction intervals.
    km: fixed exposure (frozen network scenario)
    """
    X      = sm.add_constant(monthly_df[features], has_constant='add')
    offset = np.log(np.full(len(monthly_df), km))
    mu     = model.predict(X, offset=offset)
    nb_a   = model.scale

    lower = stats.nbinom.ppf(alpha/2,   n=1/nb_a, p=1/(1 + nb_a*mu))
    upper = stats.nbinom.ppf(1-alpha/2, n=1/nb_a, p=1/(1 + nb_a*mu))

    return mu.values, lower, upper

print("=" * 60)
print("STEP 2: Loading CMIP6 projection files")
print("=" * 60)

# Load historical climate for lag computation
hist_path = os.path.join(script_dir, 'climate_monthly.csv')
if not os.path.exists(hist_path):
    raise FileNotFoundError(
        "climate_monthly.csv not found. "
        "Run build_climate.py first."
    )
hist_climate = pd.read_csv(hist_path)
print(f"  Historical climate loaded: {hist_climate.shape}")

# Load all CMIP6 monthly files
cmip6_files = glob.glob(os.path.join(script_dir, "cmip6_monthly_*.csv"))
if not cmip6_files:
    raise FileNotFoundError(
        "No cmip6_monthly_*.csv files found. "
        "Run process_cmip6.py first."
    )

print(f"  Found {len(cmip6_files)} CMIP6 files:")
for f in sorted(cmip6_files):
    df_tmp = pd.read_csv(f)
    print(f"    {os.path.basename(f)}: {len(df_tmp)} months")

print("\n" + "=" * 60)
print("STEP 3: Generating projections 2026-2100")
print("=" * 60)

all_annual = []

for fpath in sorted(cmip6_files):
    fname    = os.path.basename(fpath)
    # Parse model and scenario from filename
    # e.g. cmip6_monthly_CanESM5_ssp126.csv
    parts    = fname.replace('.csv','').split('_')
    model    = parts[2]    # CanESM5 or MIROC6
    scenario = parts[3]    # ssp126 etc

    print(f"\n  Processing {model} {scenario}...")

    proj = pd.read_csv(fpath)

    # Add sine/cosine month for CI model (cos_month is a feature)
    proj['sin_month'] = np.sin(2 * np.pi * proj['month'] / 12)
    proj['cos_month'] = np.cos(2 * np.pi * proj['month'] / 12)

    # Add lags
    proj_lagged = add_lags(proj, hist_climate)

    # Drop rows with any NaN in needed features
    ci_features_needed = CI_FEATURES
    di_features_needed = DI_FEATURES_B

    proj_ci = proj_lagged.dropna(subset=ci_features_needed).copy()
    proj_di = proj_lagged.dropna(subset=di_features_needed).copy()

    print(f"    CI projection months: {len(proj_ci)}")
    print(f"    DI projection months: {len(proj_di)}")

    # Generate monthly predictions
    ci_mu, ci_lo, ci_hi = predict_nb_monthly(model_ci, CI_FEATURES, proj_ci, KM_CI)
    di_mu, di_lo, di_hi = predict_nb_monthly(model_di, DI_FEATURES_B, proj_di, KM_DI)

    # Aggregate to annual
    proj_ci['pred_ci']    = ci_mu
    proj_ci['pred_ci_lo'] = ci_lo
    proj_ci['pred_ci_hi'] = ci_hi
    proj_ci['model']      = model
    proj_ci['scenario']   = scenario

    proj_di['pred_di']    = di_mu
    proj_di['pred_di_lo'] = di_lo
    proj_di['pred_di_hi'] = di_hi

    # Annual aggregation
    ann_ci = proj_ci.groupby('year').agg(
        breaks_ci    = ('pred_ci',    'sum'),
        breaks_ci_lo = ('pred_ci_lo', 'sum'),
        breaks_ci_hi = ('pred_ci_hi', 'sum'),
        model        = ('model',      'first'),
        scenario     = ('scenario',   'first'),
    ).reset_index()

    ann_di = proj_di.groupby('year').agg(
        breaks_di    = ('pred_di',    'sum'),
        breaks_di_lo = ('pred_di_lo', 'sum'),
        breaks_di_hi = ('pred_di_hi', 'sum'),
    ).reset_index()

    ann = ann_ci.merge(ann_di, on='year')

    # Failure rates
    ann['rate_ci']    = ann['breaks_ci']    / KM_CI
    ann['rate_ci_lo'] = ann['breaks_ci_lo'] / KM_CI
    ann['rate_ci_hi'] = ann['breaks_ci_hi'] / KM_CI
    ann['rate_di']    = ann['breaks_di']    / KM_DI
    ann['rate_di_lo'] = ann['breaks_di_lo'] / KM_DI
    ann['rate_di_hi'] = ann['breaks_di_hi'] / KM_DI

    # Pct change vs historical baseline
    ann['pct_change_ci'] = ((ann['rate_ci'] - hist_ci_rate) / hist_ci_rate * 100).round(1)
    ann['pct_change_di'] = ((ann['rate_di'] - hist_di_rate) / hist_di_rate * 100).round(1)

    all_annual.append(ann)

    # Quick sanity print
    for yr in [2030, 2050, 2070, 2090]:
        row = ann[ann['year']==yr]
        if len(row):
            r = row.iloc[0]
            print(f"    {yr}: CI={r['rate_ci']:.3f} br/km/yr ({r['pct_change_ci']:+.1f}%)  "
                  f"DI={r['rate_di']:.3f} br/km/yr ({r['pct_change_di']:+.1f}%)")

print("\n" + "=" * 60)
print("STEP 4: Compiling results")
print("=" * 60)

projections = pd.concat(all_annual, ignore_index=True)

# Save annual projections
ann_path = os.path.join(script_dir, 'projections_annual.csv')
projections.to_csv(ann_path, index=False)
print(f"  Saved: projections_annual.csv ({len(projections)} rows)")

# 30-year window summaries
summary_rows = []
for (window_name, yr_start, yr_end) in WINDOWS:
    sub = projections[(projections['year']>=yr_start) & (projections['year']<=yr_end)]
    for (model, scenario), grp in sub.groupby(['model','scenario']):
        summary_rows.append({
            'window':    window_name,
            'model':     model,
            'scenario':  scenario,
            'mean_rate_ci':     round(grp['rate_ci'].mean(), 4),
            'mean_rate_ci_lo':  round(grp['rate_ci_lo'].mean(), 4),
            'mean_rate_ci_hi':  round(grp['rate_ci_hi'].mean(), 4),
            'mean_rate_di':     round(grp['rate_di'].mean(), 4),
            'mean_rate_di_lo':  round(grp['rate_di_lo'].mean(), 4),
            'mean_rate_di_hi':  round(grp['rate_di_hi'].mean(), 4),
            'pct_change_ci':    round(grp['pct_change_ci'].mean(), 1),
            'pct_change_di':    round(grp['pct_change_di'].mean(), 1),
        })

summary_df = pd.DataFrame(summary_rows)
summ_path = os.path.join(script_dir, 'projections_30yr_summary.csv')
summary_df.to_csv(summ_path, index=False)
print(f"  Saved: projections_30yr_summary.csv ({len(summary_df)} rows)")

# Scenario spread (max-min across all model-scenario combos) per decade
projections['decade'] = (projections['year']//10)*10
spread_rows = []
for (decade, mat, rate_col, hist_rate) in [
    *[(d,'CI','rate_ci',hist_ci_rate) for d in projections['decade'].unique()],
    *[(d,'DI','rate_di',hist_di_rate) for d in projections['decade'].unique()],
]:
    sub = projections[projections['decade']==decade]
    grp = sub.groupby(['model','scenario'])[rate_col].mean()
    spread_rows.append({
        'decade':    decade,
        'material':  mat,
        'min_rate':  round(grp.min(), 4),
        'max_rate':  round(grp.max(), 4),
        'spread':    round(grp.max()-grp.min(), 4),
        'hist_rate': round(hist_rate, 4),
    })

spread_df = pd.DataFrame(spread_rows).sort_values(['material','decade'])
spread_path = os.path.join(script_dir, 'projections_scenario_spread.csv')
spread_df.to_csv(spread_path, index=False)
print(f"  Saved: projections_scenario_spread.csv")

print("\n" + "=" * 60)
print("STEP 5: Results summary")
print("=" * 60)

print(f"\n  Historical baseline (1997-2025):")
print(f"    CI: {hist_ci_rate:.4f} breaks/km/year")
print(f"    DI: {hist_di_rate:.4f} breaks/km/year")

print(f"\n  30-year window mean rates (breaks/km/year):")
print(f"\n  {'Window':<12} {'Model':<10} {'Scenario':<10} "
      f"{'CI rate':>9} {'CI chg%':>8} {'DI rate':>9} {'DI chg%':>8}")
print("  " + "-"*68)

for _, row in summary_df.sort_values(['window','model','scenario']).iterrows():
    print(f"  {row['window']:<12} {row['model']:<10} {row['scenario']:<10} "
          f"{row['mean_rate_ci']:>9.4f} {row['pct_change_ci']:>7.1f}% "
          f"{row['mean_rate_di']:>9.4f} {row['pct_change_di']:>7.1f}%")

print(f"\n  Scenario spread (max-min across all combos):")
print(f"  {'Decade':<8} {'CI spread':>10} {'DI spread':>10}")
for decade in sorted(projections['decade'].unique()):
    ci_sp = spread_df[(spread_df['decade']==decade)&(spread_df['material']=='CI')]['spread'].values
    di_sp = spread_df[(spread_df['decade']==decade)&(spread_df['material']=='DI')]['spread'].values
    if len(ci_sp) and len(di_sp):
        print(f"  {decade:<8} {ci_sp[0]:>10.4f} {di_sp[0]:>10.4f}")

print("\n  FTC WARNING:")
print("  CanESM5 projects 13-14 FTC/month in January vs observed 8.5.")
print("  DI projections using FTC features may be inflated.")
print("  Flag this as a limitation in the paper.")

print("\n" + "=" * 60)
print("DONE")
print("=" * 60)
print("""
Output files:
  projections_annual.csv         -- year-by-year rates for all 8 scenarios
  projections_30yr_summary.csv   -- 30-year window means (2026-55, 2041-70, 2071-2100)
  projections_scenario_spread.csv-- scenario uncertainty spread per decade

""")

import os, warnings
import numpy as np
import pandas as pd
import statsmodels.api as sm
from statsmodels.genmod.generalized_linear_model import GLM
from statsmodels.genmod import families
import scipy.stats as stats
warnings.filterwarnings('ignore')

try:
    script_dir = os.path.dirname(os.path.abspath(__file__))
except NameError:
    script_dir = os.getcwd()

def p(msg): print(f"\n{'='*60}\n{msg}\n{'='*60}")

def fit_nb(df, features, offset='km_primary'):
    X = sm.add_constant(df[features])
    y = df['break_count']
    off = np.log(df[offset].clip(lower=0.001))
    return GLM(y, X, family=families.NegativeBinomial(alpha=1.0),
               offset=off).fit(maxiter=200, disp=False)

def pred_nb(mod, df, features, offset='km_primary', alpha=0.10):
    X   = sm.add_constant(df[features], has_constant='add')
    off = np.log(df[offset].clip(lower=0.001))
    mu  = mod.predict(X, offset=off)
    a   = mod.scale
    lo  = stats.nbinom.ppf(alpha/2,   n=1/a, p=1/(1+a*mu))
    hi  = stats.nbinom.ppf(1-alpha/2, n=1/a, p=1/(1+a*mu))
    return mu.values, lo, hi

def annual_rate_mae(pred_df, km_col='km_primary'):
    ann = pred_df.groupby('year').agg(
        actual=('break_count','sum'),
        predicted=('predicted','sum'),
        km=(km_col,'mean')).reset_index()
    ann['act_rate']  = ann['actual']    / ann['km']
    ann['pred_rate'] = ann['predicted'] / ann['km']
    return round(float(np.mean(np.abs(ann['act_rate']-ann['pred_rate']))),4), ann

def mase(actual, predicted, baseline_mae):
    return round(float(np.mean(np.abs(actual-predicted)))/baseline_mae, 3)

def coverage90(actual, lo, hi):
    return round(float(np.mean((actual>=lo)&(actual<=hi)))*100, 1)

p("STEP 1: Loading data")

---
## Step 7: DI Pipe-Level Supporting Analysis

Fits a negative binomial model at pipe-year level for ductile iron pipes:

`breaks ~ age + age² + prior_breaks, offset = log(pipe_length_km)`

This analysis uses age variation **within** each calendar year to separate age from calendar time — breaking the age-calendar confounding present at cohort level.

**Key finding:** Each prior break multiplies DI failure rate by **1.76×** (p < 0.001). Prior break history is a stronger deterioration signal than pipe age.

**CI ageing note:** The same analysis for CI is not reported because the CI age coefficient is not statistically significant (p = 0.91) due to survivorship bias in the closed pre-1970 cohort. This age-period-cohort identification problem is documented as a methodological finding.


In [ ]:

d  = pd.read_csv(os.path.join(script_dir, 'modelling_dataset.csv'))
b  = pd.read_csv(os.path.join(script_dir, 'Water_Main_Breaks.csv'),
                 encoding='utf-8-sig', low_memory=False)
m  = pd.read_csv(os.path.join(script_dir, 'Water_Mains.csv'),
                 encoding='utf-8-sig', low_memory=False)
pr = pd.read_csv(os.path.join(script_dir, 'projections_annual.csv'))

b['dt'] = pd.to_datetime(b['Incident date'], errors='coerce')
b['yr'] = b['dt'].dt.year
m['iy'] = pd.to_datetime(m['INSTALLATION_DATE'], errors='coerce').dt.year

print(f"  modelling_dataset: {d.shape}")
print(f"  breaks: {len(b):,}  |  mains: {len(m):,}  |  projections: {len(pr):,}")

p("STEP 2: Fitting negative binomial models (Split A and B)")

# Features selected by stepwise AIC (confirmed from notebook)
CI_FEAT = ['FI', 'Tmean_lag1', 'FD_lag1', 'cos_month']
DI_A_FEAT = ['FI', 'Tmean_lag2', 'FTC_lag1', 'FTC_lag3', 'FD_lag2']
DI_B_FEAT = ['FI', 'FI_lag2', 'FD_lag2']

results_rows = []

for mat, feat_A, feat_B in [('CI', CI_FEAT, CI_FEAT),
                              ('DI', DI_A_FEAT, DI_B_FEAT)]:
    md = d[d['material']==mat].copy()

    # Split A
    trA  = md[md['year']<=2016]
    vaA  = md[(md['year']>=2017)&(md['year']<=2021)]
    teA  = md[md['year']>=2022]
    trvaA= md[md['year']<=2021]

    # M0 baseline for Split A
    m0A = trA.groupby('month_num')['break_count'].mean()
    m0_mae_teA = float(np.mean(np.abs(teA['break_count'].values -
                                       teA['month_num'].map(m0A).values)))

    # Fit Split A — refit on train+val for test evaluation
    modA = fit_nb(trvaA, feat_A)
    pA, loA, hiA = pred_nb(modA, teA, feat_A)
    predA = teA.copy(); predA['predicted']=pA; predA['lower_90']=loA; predA['upper_90']=hiA
    predA['residual'] = teA['break_count'].values - pA
    mae_rate_A, _ = annual_rate_mae(predA)
    results_rows.append({'material':mat,'split':'A','model':'M2_stepwise',
        'train_years':'1997-2016','test_years':'2022-2025',
        'MAE_monthly':round(float(np.mean(np.abs(teA['break_count'].values-pA))),3),
        'MAE_annual':round(float(np.mean(np.abs(
            teA.groupby('year')['break_count'].sum().values -
            pd.Series(pA,index=teA.index).groupby(teA['year']).sum().values))),1),
        'MAE_rate':mae_rate_A,
        'MASE':mase(teA['break_count'].values, pA, m0_mae_teA),
        'Coverage_90':coverage90(teA['break_count'].values, loA, hiA),
        'Bias':round(float(np.mean(pA-teA['break_count'].values)),3)})

    # Save coefficients Split A
    coef_A = pd.DataFrame({'feature':modA.params.index,
        'coefficient':modA.params.values.round(5),
        'std_error':modA.bse.values.round(5),
        'p_value':modA.pvalues.values.round(4),
        'significant':modA.pvalues.values<0.05})
    coef_A.to_csv(os.path.join(script_dir,
        f'coefficients_{mat.lower()}_splitA.csv'), index=False)

    # Save predictions Split A
    predA[['material','month_str','year','month_num','break_count',
           'km_primary','predicted','lower_90','upper_90','residual']
          ].to_csv(os.path.join(script_dir,
        f'predictions_{mat.lower()}_splitA.csv'), index=False)

    print(f"  {mat} Split A: MAE_rate={mae_rate_A}  "
          f"MASE={results_rows[-1]['MASE']}")

    # Split B
    trB = md[md['year']<=2020]
    teB = md[md['year']>=2021]

    # M0 baseline for Split B
    m0B = trB.groupby('month_num')['break_count'].mean()
    m0_mae_teB = float(np.mean(np.abs(teB['break_count'].values -
                                       teB['month_num'].map(m0B).values)))

    modB = fit_nb(trB, feat_B)
    pB, loB, hiB = pred_nb(modB, teB, feat_B)
    predB = teB.copy(); predB['predicted']=pB; predB['lower_90']=loB; predB['upper_90']=hiB
    predB['residual'] = teB['break_count'].values - pB
    mae_rate_B, _ = annual_rate_mae(predB)
    results_rows.append({'material':mat,'split':'B','model':'M2_stepwise',
        'train_years':'1997-2020','test_years':'2021-2025',
        'MAE_monthly':round(float(np.mean(np.abs(teB['break_count'].values-pB))),3),
        'MAE_annual':round(float(np.mean(np.abs(
            teB.groupby('year')['break_count'].sum().values -
            pd.Series(pB,index=teB.index).groupby(teB['year']).sum().values))),1),
        'MAE_rate':mae_rate_B,
        'MASE':mase(teB['break_count'].values, pB, m0_mae_teB),
        'Coverage_90':coverage90(teB['break_count'].values, loB, hiB),
        'Bias':round(float(np.mean(pB-teB['break_count'].values)),3)})

    # Save coefficients Split B
    coef_B = pd.DataFrame({'feature':modB.params.index,
        'coefficient':modB.params.values.round(5),
        'std_error':modB.bse.values.round(5),
        'p_value':modB.pvalues.values.round(4),
        'significant':modB.pvalues.values<0.05})
    coef_B.to_csv(os.path.join(script_dir,
        f'coefficients_{mat.lower()}_splitB.csv'), index=False)

    # Save predictions Split B
    predB[['material','month_str','year','month_num','break_count',
           'km_primary','predicted','lower_90','upper_90','residual']
          ].to_csv(os.path.join(script_dir,
        f'predictions_{mat.lower()}_splitB.csv'), index=False)

    print(f"  {mat} Split B: MAE_rate={mae_rate_B}  "
          f"MASE={results_rows[-1]['MASE']}")

# Save model results
results_df = pd.DataFrame(results_rows)
results_df.to_csv(os.path.join(script_dir,'model_results_both_splits.csv'), index=False)
print(f"\n  Saved: model_results_both_splits.csv")
print(results_df[['material','split','MAE_rate','MASE','Coverage_90','Bias']
                 ].to_string(index=False))

p("STEP 3: DI pipe-level model (age + prior breaks)")

---
## Step 8: DI Design Life Analysis

Simulates whether each active DI pipe experiences a first break before its 75-year design life (AWWA 2012) within the 2025–2050 planning horizon.

**Method:** Analytical survival simulation per pipe:
- Hazard each year = 1 − exp(−rate × pipe_length_km)
- Rate = pipe-level model rate × climate multiplier from Step 6
- Survival S(t) = ∏(1 − hazard) up to year t

**Key findings:**
- Historical baseline: **11.2%** of DI pipes experience first break before design life by 2050
- Under warming scenarios: **9.4–10.3%** (modest reduction from frost relief)
- Average age at first break: **57 years** (~18 years before design life)
- Climate has limited influence on the **timing** of first break; it affects **how many** pipes fail

**Wording note:** First break ≠ end of life. A pipe that breaks at 57 years is repaired and continues operating.


In [ ]:

di_mains = m[(m['MATERIAL']=='DI')&(m['Shape__Length']>0)&(m['iy'].notna())].copy()
bm = b[(b['Type of Asset Broken']=='MAIN')&
       (b.yr>=1997)&(b.yr<=2025)&
       (b['Asset Material']=='DI')].copy()
bm['aid'] = pd.to_numeric(bm['Related Asset ID'], errors='coerce')
bm_s = bm.sort_values(['aid','dt'])
bm_s['prior'] = bm_s.groupby('aid').cumcount()

linked = bm_s[bm_s['aid'].isin(di_mains['WATMAINID'])].merge(
    di_mains[['WATMAINID','iy','Shape__Length']],
    left_on='aid', right_on='WATMAINID', how='left')
linked['age'] = linked['yr'] - linked['iy']
linked = linked[(linked['age']>=0)&(linked['age']<=120)].copy()

brk_cnt = linked.groupby(['WATMAINID','yr']).size().reset_index(name='breaks')
prior_lu = linked.sort_values(['WATMAINID','yr']).groupby(
    ['WATMAINID','yr'])['prior'].min().reset_index()

rows = []
for _, pipe in di_mains.iterrows():
    pid = pipe['WATMAINID']; iy = int(pipe['iy'])
    length_km = pipe['Shape__Length']/1000.0
    for yr in range(max(1997,iy), 2026):
        age = yr - iy
        br = brk_cnt[(brk_cnt['WATMAINID']==pid)&(brk_cnt['yr']==yr)]
        n_b = int(br['breaks'].values[0]) if len(br) else 0
        pb = prior_lu[(prior_lu['WATMAINID']==pid)&(prior_lu['yr']<yr)]
        pn = int(pb['prior'].max()+1) if len(pb) else 0
        rows.append({'pipe_id':pid,'year':yr,'age':age,
                     'prior_breaks':pn,'breaks':n_b,'length_km':length_km})

panel = pd.DataFrame(rows)
panel = panel[(panel['age']>=0)&(panel['length_km']>0)].copy()
panel['log_length']       = np.log(panel['length_km'])
panel['prior_breaks_cap'] = panel['prior_breaks'].clip(upper=5)
panel['age_sq']           = panel['age']**2 / 1000

X   = sm.add_constant(panel[['age','age_sq','prior_breaks_cap']])
mod_di_pipe = GLM(panel['breaks'], X,
                  family=families.NegativeBinomial(alpha=1.0),
                  offset=panel['log_length']).fit(maxiter=200, disp=False)

c_const = mod_di_pipe.params['const']
c_age   = mod_di_pipe.params['age']
c_agesq = mod_di_pipe.params['age_sq']
c_prior = mod_di_pipe.params['prior_breaks_cap']

print(f"  AIC: {mod_di_pipe.aic:.1f}")
print(f"  Coefficients:")
for f, c, p_v in zip(mod_di_pipe.params.index,
                      mod_di_pipe.params.values,
                      mod_di_pipe.pvalues.values):
    print(f"    {f:22s}  coef={c:+.5f}  p={p_v:.4f}")

# Rate multipliers
print(f"\n  Rate multipliers by prior break count:")
multipliers = []
for pb in range(6):
    mult = round(float(np.exp(c_prior * pb)), 2)
    print(f"    {pb} prior breaks: {mult}x")
    multipliers.append({'prior_breaks':pb,'rate_multiplier':mult})

# Peak age
peak_age = round(-c_age * 1000 / (2*c_agesq), 1)
print(f"\n  Age at peak rate: {peak_age} years")

# Save coefficients
coef_pipe = pd.DataFrame({'feature':mod_di_pipe.params.index,
    'coefficient':mod_di_pipe.params.values.round(5),
    'std_error':mod_di_pipe.bse.values.round(5),
    'p_value':mod_di_pipe.pvalues.values.round(4),
    'significant':mod_di_pipe.pvalues.values<0.05})
coef_pipe.to_csv(os.path.join(script_dir,'di_pipe_level_coefficients.csv'), index=False)

# Save summary
summary_pipe = pd.DataFrame([
    {'finding':'Prior break multiplier per additional break',
     'value':round(float(np.exp(c_prior)),3),
     'note':'Each prior break multiplies rate by this factor'},
    {'finding':'Age at peak break rate (years)',
     'value':peak_age,
     'note':'Beyond this age, survivorship bias dominates'},
    {'finding':'Total pipe-years in panel','value':len(panel),
     'note':'Sample size'},
    {'finding':'Total DI breaks in panel','value':int(panel['breaks'].sum()),
     'note':'Total events'},
    {'finding':'Model AIC','value':round(mod_di_pipe.aic,1),'note':''},
] + [{'finding':f'{r["prior_breaks"]} prior breaks → rate multiplier',
      'value':r['rate_multiplier'],'note':''} for r in multipliers])
summary_pipe.to_csv(os.path.join(script_dir,'di_pipe_level_summary.csv'), index=False)

print(f"\n  Saved: di_pipe_level_coefficients.csv")
print(f"  Saved: di_pipe_level_summary.csv")

p("STEP 4: DI design life analysis (75 years, horizon 2050)")

DESIGN_LIFE_DI = 75
START_YEAR     = 2025
HORIZON_YEAR   = 2050
KM_DI          = 322.05
HIST_DI_RATE   = 0.0663

di = di_mains.copy()
di['age_2025']   = START_YEAR - di['iy']
di['dl_year']    = di['iy'] + DESIGN_LIFE_DI
di['length_km'] = di['Shape__Length'] / 1000.0

prior_2025 = panel[panel['year']==2025][['pipe_id','prior_breaks']].rename(
    columns={'prior_breaks':'pb_2025'})
di = di.merge(prior_2025, left_on='WATMAINID', right_on='pipe_id', how='left')
di['pb_2025'] = di['pb_2025'].fillna(0).clip(upper=5).astype(int)
di_sim = di[(di['dl_year'] > START_YEAR) & (di['age_2025'] >= 0)].copy()

print(f"  DI pipes in simulation: {len(di_sim):,}")
print(f"  Of which reach DL before 2050: "
      f"{((di_sim['dl_year']>START_YEAR)&(di_sim['dl_year']<=HORIZON_YEAR)).sum()}")

def pipe_rate_fn(age, prior):
    return np.exp(c_const + c_age*age + c_agesq*age**2/1000 + c_prior*min(prior,5))

# Historical baseline rate per pipe in 2025
baseline_rates = di_sim.apply(
    lambda r: pipe_rate_fn(r['age_2025'], r['pb_2025']), axis=1)
baseline_mean = baseline_rates.mean()

dl_results  = []
pipe_preds  = []

scenarios = [('historical', None, None)] + [
    (f"{gcm}_{ssp}", gcm, ssp)
    for gcm in ['CanESM5','MIROC6']
    for ssp in ['ssp126','ssp245','ssp370','ssp585']
]

for (label, gcm, ssp) in scenarios:
    if label == 'historical':
        clim_mult = {yr: 1.0 for yr in range(2026, HORIZON_YEAR+1)}
    else:
        sub = pr[(pr['model']==gcm)&(pr['scenario']==ssp)]
        base_rate = HIST_DI_RATE
        clim_mult = dict(zip(sub['year'],
                              sub['rate_di'].values / base_rate))

    scen_rows = []
    for _, pipe in di_sim.iterrows():
        age_now  = int(pipe['age_2025'])
        prior_now= int(pipe['pb_2025'])
        dl_yr    = int(pipe['dl_year'])
        sim_yrs  = int(min(dl_yr, HORIZON_YEAR) - START_YEAR)
        if sim_yrs <= 0:
            continue

        years_arr    = np.arange(1, sim_yrs+1)
        yrs_cal      = START_YEAR + years_arr
        ages_arr     = age_now + years_arr
        log_rates    = (c_const + c_age*ages_arr +
                        c_agesq*ages_arr**2/1000 + c_prior*min(prior_now,5))
        base_rates_yr= np.exp(log_rates)
        clim_arr     = np.array([clim_mult.get(int(y), 1.0) for y in yrs_cal])
        rates_yr     = base_rates_yr * clim_arr

        hazard = 1 - np.exp(-rates_yr * pipe['length_km'])
        surv   = np.concatenate([[1.0], np.cumprod(1-hazard)])
        p_fail_yr = surv[:-1] * hazard

        mask_dl = yrs_cal <= dl_yr
        prob_before_dl  = float(p_fail_yr[mask_dl].sum())
        prob_by_horizon = float(p_fail_yr.sum())

        if prob_by_horizon > 0:
            mean_fail_yr = float(np.sum(yrs_cal*p_fail_yr)/prob_by_horizon)
            years_lost   = dl_yr - mean_fail_yr
        else:
            mean_fail_yr = np.nan
            years_lost   = np.nan

        scen_rows.append({'scenario':label,
            'pipe_id':pipe['WATMAINID'],
            'age_2025':age_now,'dl_year':dl_yr,
            'p_fail_before_dl':round(prob_before_dl,4),
            'p_fail_by_2050':round(prob_by_horizon,4),
            'mean_failure_year':round(mean_fail_yr,1) if not np.isnan(mean_fail_yr) else np.nan,
            'years_lost':round(years_lost,1) if not np.isnan(years_lost) else np.nan})

    scen_df = pd.DataFrame(scen_rows)
    pipe_preds.append(scen_df)

    pct_before_dl = scen_df['p_fail_before_dl'].mean()*100
    pct_by_2050   = scen_df['p_fail_by_2050'].mean()*100
    mean_yrs_lost = scen_df['years_lost'].mean()

    print(f"  {label:<25}  %fail before DL={pct_before_dl:.2f}%  "
          f"yrs lost={mean_yrs_lost:.1f}")

    dl_results.append({'scenario':label,
        'pct_fail_before_dl':round(pct_before_dl,2),
        'pct_fail_by_2050':round(pct_by_2050,2),
        'mean_years_lost':round(mean_yrs_lost,2)})

dl_df = pd.DataFrame(dl_results)
dl_df.to_csv(os.path.join(script_dir,'design_life_results.csv'), index=False)

pipe_pred_all = pd.concat(pipe_preds, ignore_index=True)
pipe_pred_all.to_csv(os.path.join(script_dir,
    'design_life_pipe_predictions.csv'), index=False)

print(f"\n  Saved: design_life_results.csv")
print(f"  Saved: design_life_pipe_predictions.csv")


# ── 5. PRINT COMPLETE RESULTS SUMMARY ────────────────────────────────────────
p("COMPLETE RESULTS SUMMARY FOR REPORT")

print("\n  TABLE 1: Model Performance")
print(results_df[['material','split','train_years','test_years',
                   'MAE_rate','MASE','Coverage_90','Bias']].to_string(index=False))

print("\n  TABLE 2: CI Coefficients (Split B)")
ci_coef = pd.read_csv(os.path.join(script_dir,'coefficients_ci_splitB.csv'))
print(ci_coef.to_string(index=False))

print("\n  TABLE 3: Projection summary (CI, end of century)")
proj30 = pd.read_csv(os.path.join(script_dir,'projections_30yr_summary.csv'))
eoc = proj30[proj30['window']=='2071-2100']
print(eoc[['model','scenario','mean_rate_ci','pct_change_ci',
           'mean_rate_di','pct_change_di']].to_string(index=False))

print("\n  DI PIPE-LEVEL KEY NUMBERS:")
print(f"    Prior break multiplier: {float(np.exp(c_prior)):.2f}x per break")
print(f"    Age peaks at:           {peak_age} years")
print(f"    p-value (prior_breaks): {mod_di_pipe.pvalues['prior_breaks_cap']:.4f}")

print("\n  TABLE 4: Design Life Results")
print(dl_df.to_string(index=False))

print("\n" + "="*60)
print("ALL DONE")
print("="*60)
print("""
Files produced:
  model_results_both_splits.csv    → Table 1
  coefficients_ci_splitA/B.csv    → Table 2
  coefficients_di_splitA/B.csv    → DI model
  predictions_ci/di_splitA/B.csv  → Figure 3
  di_pipe_level_coefficients.csv  → DI pipe-level
  di_pipe_level_summary.csv       → DI pipe-level + Figure 4
  design_life_results.csv         → Table 4
  design_life_pipe_predictions.csv→ supporting data


---
## Results Summary

All output files are saved to the working directory. Key files for the report:

| File | Report use |
|---|---|
| `model_results_both_splits.csv` | Table 1 — Model performance |
| `coefficients_ci_splitB.csv` | Table 2 — CI coefficients (primary) |
| `projections_30yr_summary.csv` | Table 3 — Projected CI change |
| `di_pipe_level_summary.csv` | DI pipe-level paragraph |
| `design_life_results.csv` | Table 4 — Design life results |

## Citation

If using this analysis, please cite:

> Khashei, M., Boloukasli ahmadgourabi, F., and Dziedzic, R. (2024). Predicting the Future Failures of Urban Water Systems. *Eng. Proc.*, 69, 35. *(extended by this work)*

## License

Open for academic use. Data from City of Kitchener Open Data Portal and Environment and Climate Change Canada.
